# 1) Pre & Post Intervention Block Analysis

## 1.1) MVC and 9HTP Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ==============================================================================
# 0. Función de Generación de Gráficas de Validación (Pre/Post y Diferencias)
# ==============================================================================
def generate_validation_plots(df_pivot, output_dir):
    """
    Genera las gráficas de validación clínicas (Valores Pre/Post arriba y 
    diferencias Post - Pre abajo) para MVC y 9HPT, guardándolas en 'Validation_Plots'.
    """
    fig_dir = os.path.join(output_dir, "Validation_Plots")
    os.makedirs(fig_dir, exist_ok=True)
    
    metricas_info = [
        ('MVC', 'MVC', 'MVC (Kg)', 'MVC post - pre (Kg)', 3.0, 0.6),
        ('9HPT_Derecha', '9HPT Right', '9HPT (s)', '9HPT post - pre (s)', 1.0, 0.6),
        ('9HPT_Izquierda', '9HPT Left', '9HPT (s)', '9HPT post - pre (s)', 1.0, 0.6)
    ]
    
    for col, title, ylabel_top, ylabel_bot, offset_top, offset_bot in metricas_info:
        if col not in df_pivot.columns:
            continue
            
        # Pivotar temporalmente para tener PRE y POST como columnas por sujeto/sesión
        sub_df = df_pivot.pivot(index=['Subject', 'Stimulation'], columns='Time', values=col).dropna()
        
        stim_pre = sub_df.loc[sub_df.index.get_level_values('Stimulation') == 'STIM', 'PRE'].values
        stim_post = sub_df.loc[sub_df.index.get_level_values('Stimulation') == 'STIM', 'POST'].values
        sham_pre = sub_df.loc[sub_df.index.get_level_values('Stimulation') == 'SHAM', 'PRE'].values
        sham_post = sub_df.loc[sub_df.index.get_level_values('Stimulation') == 'SHAM', 'POST'].values
        
        fig, axs = plt.subplots(2, 1, figsize=(6, 8))
        text_offset = -0.075
        
        # --- Subplot Superior: PRE vs POST ---
        axs[0].set_title(title, fontsize=16, fontweight='bold')
        axs[0].set_ylabel(ylabel_top, fontsize=14)
        x_loc_top = [0.5, 1.0, 2.0, 2.5]
        bp0 = axs[0].boxplot([stim_pre, stim_post, sham_pre, sham_post], positions=x_loc_top, widths=0.4)
        axs[0].tick_params(axis='x', which='both', bottom=False, labelbottom=False)
        axs[0].set_ylim(ymin=axs[0].get_ylim()[0] - offset_top)
        
        caps0 = [bp0['caps'][i].get_ydata()[0] for i in [0, 2, 4, 6]]
        for x, cap, lbl in zip(x_loc_top, caps0, ['pre', 'post', 'pre', 'post']):
            axs[0].text(x, cap - offset_top, lbl, ha='center', va='bottom', fontsize=9, fontweight='bold')
            
        axs[0].text(0.75, text_offset, 'STIM', ha='center', va='bottom', fontsize=9, fontweight='bold', transform=axs[0].get_xaxis_transform(), clip_on=False)
        axs[0].text(2.25, text_offset, 'SHAM', ha='center', va='bottom', fontsize=9, fontweight='bold', transform=axs[0].get_xaxis_transform(), clip_on=False)
        
        # --- Subplot Inferior: Diferencia POST - PRE ---
        axs[1].set_ylabel(ylabel_bot, fontsize=14)
        x_loc_bot = [1.0, 2.0]
        bp1 = axs[1].boxplot([stim_post - stim_pre, sham_post - sham_pre], positions=x_loc_bot, widths=0.4)
        axs[1].tick_params(axis='x', which='both', bottom=False, labelbottom=False)
        axs[1].set_ylim(ymin=axs[1].get_ylim()[0] - offset_bot)
        
        caps1 = [bp1['caps'][0].get_ydata()[0], bp1['caps'][2].get_ydata()[0]]
        for x, cap, grp in zip(x_loc_bot, caps1, ['STIM', 'SHAM']):
            axs[1].text(x, cap - offset_bot, 'post-pre', ha='center', va='bottom', fontsize=9, fontweight='bold')
            axs[1].text(x, text_offset, grp, ha='center', va='bottom', fontsize=9, fontweight='bold', transform=axs[1].get_xaxis_transform(), clip_on=False)
            
        plt.tight_layout()
        fig.savefig(os.path.join(fig_dir, f"Val_{col}.png"), dpi=300, bbox_inches='tight')
        plt.close(fig)

# ==============================================================================
# 1. Función de Extracción Matemática (Promediado y Formato Largo)
# ==============================================================================
def extract_metrics(df_raw):
    """
    Limpia el DataFrame crudo del Excel, promedia las repeticiones de 9HPT
    y transforma la tabla a formato largo (Long Format).
    """
    df_raw = df_raw.dropna(subset=['Subject']).copy()
    df_raw['Subject'] = df_raw['Subject'].astype(int).astype(str)
    df_raw['Stimulation'] = df_raw['Stimulation'].str.strip().str.upper()
    
    # Asegurar tipo numérico en todas las mediciones
    cols_num = df_raw.columns[2:]
    df_raw[cols_num] = df_raw[cols_num].apply(pd.to_numeric, errors='coerce')
    
    # Promedios PRE y POST de 9HPT
    df_raw['PRE_9HPT_Derecha'] = df_raw[['Pre_9HPT_D1', 'Pre_9HPT_D2']].mean(axis=1)
    df_raw['PRE_9HPT_Izquierda'] = df_raw[['Pre_9HPT_I1', 'Pre_9HPT_I2']].mean(axis=1)
    df_raw['POST_9HPT_Derecha'] = df_raw[['Post_9HPT_D1', 'Post_9HPT_D2']].mean(axis=1)
    df_raw['POST_9HPT_Izquierda'] = df_raw[['Post_9HPT_I1', 'Post_9HPT_I2']].mean(axis=1)
    
    df_raw = df_raw.rename(columns={'Pre_MVC': 'PRE_MVC', 'Post_MVC': 'POST_MVC'})
    
    columnas_finales = [
        'Subject', 'Stimulation', 'PRE_MVC', 'POST_MVC',
        'PRE_9HPT_Derecha', 'POST_9HPT_Derecha',
        'PRE_9HPT_Izquierda', 'POST_9HPT_Izquierda'
    ]
    df_proc = df_raw[columnas_finales].copy()
    
    # Transformar a formato largo
    df_melted = df_proc.melt(id_vars=['Subject', 'Stimulation'], var_name='Time_Metric', value_name='Valor')
    df_melted[['Time', 'Metric']] = df_melted['Time_Metric'].str.split('_', n=1, expand=True)
    
    df_pivot = df_melted.pivot_table(
        index=['Subject', 'Stimulation', 'Time'],
        columns='Metric',
        values='Valor',
        aggfunc='first'
    ).reset_index()
    
    return df_pivot

# ==============================================================================
# 2. Pipeline Principal (Con Filtro de Sujetos)
# ==============================================================================
def process_metric(excel_filepath, output_csv, sujetos_a_procesar=None, plot_validation=False):
    print("=" * 80)
    print(f"CARGANDO REGISTRO CLÍNICO DESDE: {excel_filepath}")
    print("=" * 80)
    
    try:
        columnas_interes = [0, 1, 2, 3, 4, 5, 6, 22, 23, 24, 25, 26]
        nombres_columnas = [
            'Subject', 'Stimulation', 'Pre_MVC', 
            'Pre_9HPT_D1', 'Pre_9HPT_D2', 'Pre_9HPT_I1', 'Pre_9HPT_I2', 
            'Post_9HPT_D1', 'Post_9HPT_D2', 'Post_9HPT_I1', 'Post_9HPT_I2', 
            'Post_MVC'
        ]
        df_raw = pd.read_excel(excel_filepath, sheet_name=0, usecols=columnas_interes, skiprows=2, names=nombres_columnas)
    except Exception as e:
        print(f"Error al cargar el archivo: {e}")
        return

    df_pivot = extract_metrics(df_raw)
    
    if sujetos_a_procesar is not None:
        df_pivot = df_pivot[df_pivot['Subject'].isin(sujetos_a_procesar)].copy()
        
    # Filtrar sujetos con diseño Cross-Over completo (4 mediciones sin NaNs)
    df_pivot = df_pivot.dropna(subset=['MVC', '9HPT_Derecha', '9HPT_Izquierda']).copy()
    conteo_sujetos = df_pivot.groupby('Subject').size()
    sujetos_completos = conteo_sujetos[conteo_sujetos == 4].index.tolist()
    
    df_final = df_pivot[df_pivot['Subject'].isin(sujetos_completos)].copy()
    df_final['Time'] = pd.Categorical(df_final['Time'], categories=['PRE', 'POST'], ordered=True)
    df_final = df_final.sort_values(by=['Subject', 'Stimulation', 'Time'])
    
    # Generar gráficas de validación (Post - Pre)
    base_out_dir = os.path.dirname(os.path.abspath(output_csv)) or os.getcwd()
    if plot_validation:
        generate_validation_plots(df_final, base_out_dir)
    
    df_final.to_csv(output_csv, index=False)
    print(f"\n¡Éxito! Matriz clínica exportada a: {output_csv}")
    print("Gráficas de validación guardadas en 'Validation_Plots'")
    print("\nPrevisualización de los datos tabulados:")
    print(df_final.head(8).to_string(index=False))

# ==============================================================================
# EJECUCIÓN
# ==============================================================================
if __name__ == '__main__':
    ARCHIVO_EXCEL = r"E:\Yoyi\Raw Data\StepByStep.xlsx"
    ARCHIVO_SALIDA_CSV = "Datos_Clinicos_MVC_9HTP.csv"

    generate_validation = False  # Cambiar a False si no se desean las gráficas de validación
    
    process_metric(ARCHIVO_EXCEL, ARCHIVO_SALIDA_CSV, plot_validation=generate_validation)

### Statistical Analysis

In [ ]:
import pandas as pd
import pingouin as pg
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ==============================================================================
# FUNCIÓN DE GRAFICADO (ESTÉTICA MINIMALISTA Y ORDENADA)
# ==============================================================================
def graficar_boxplot(df, metrica, titulo, ylabel):
    fig = plt.figure(figsize=(6, 4))
    
    ax = sns.boxplot(
        data=df, x='Stimulation', y=metrica, hue='Time',
        order=['STIM', 'SHAM'], hue_order=['PRE', 'POST'],
        color='white', width=0.5, gap=0.15, linewidth=1.5,
        fliersize=6, medianprops=dict(linewidth=2.5)
    )
    
    for patch in ax.patches:
        patch.set_edgecolor('black')
        patch.set_facecolor('white')

    lineas_medianas = [line for line in ax.lines if line.get_linewidth() == 2.5]
    medianas_ordenadas = sorted(lineas_medianas, key=lambda l: sum(l.get_xdata()) / 2)
    colores_estandar = ['#0072B2', '#D55E00', '#0072B2', '#D55E00']
    
    for mediana, color in zip(medianas_ordenadas, colores_estandar):
        mediana.set_color(color)

    plt.title(titulo, fontsize=18, fontweight='bold')
    plt.ylabel(ylabel, fontsize=14, fontweight='bold')
    plt.xlabel('')
    plt.xticks(fontsize=12, fontweight='bold')
    
    if ax.legend_:
        ax.legend_.remove()
        
    y_min, y_max = df[metrica].min(), df[metrica].max()
    rango = y_max - y_min
    offset_y = rango * 0.08
    
    cajas = ax.patches[:4]
    cajas_ordenadas = sorted(cajas, key=lambda p: p.get_path().get_extents().bounds[0])
    posiciones_x = [p.get_path().get_extents().bounds[0] + (p.get_path().get_extents().bounds[2] / 2) for p in cajas_ordenadas]
    etiquetas = ['pre', 'post', 'pre', 'post']
    
    for x, etiqueta, color_txt in zip(posiciones_x, etiquetas, colores_estandar):
        ax.text(x, y_min - offset_y, etiqueta, ha='center', va='top', fontweight='bold', fontsize=11, color=color_txt)
    
    plt.ylim(y_min - (rango * 0.15), y_max + (rango * 0.05))
    plt.tight_layout()
    
    output_dir = os.path.join(os.getcwd(), "output")
    os.makedirs(output_dir, exist_ok=True)
    fig.savefig(os.path.join(output_dir, f"{metrica}.png"), dpi=600, bbox_inches='tight')
    plt.show()
    plt.close(fig)

# ==============================================================================
# ANÁLISIS ESTADÍSTICO UNIFICADO
# ==============================================================================
def run_statistical_pipeline(csv_filepath, config_metricas, usar_log=False, plot_validation=False):
    print("=" * 90)
    print(f"CARGANDO DATOS DESDE: {csv_filepath} | Bloque: Clínico (MVC & 9HPT)")
    print("=" * 90)
    
    try:
        df = pd.read_csv(csv_filepath)
    except FileNotFoundError:
        print(f"Error: Archivo '{csv_filepath}' no encontrado.")
        return
        
    df['Subject'] = df['Subject'].astype(str)
    df['Stimulation'] = df['Stimulation'].astype(str)
    df['Time'] = df['Time'].astype(str)

    # Exclusión manual (Homologado: puedes dejar ['1', '2'] o ['1', '2', '10'])
    sujetos_excluidos_manual = ['1', '2', '10']
    df = df[~df['Subject'].isin(sujetos_excluidos_manual)]
    
    conteo_mediciones = df.groupby('Subject').size()
    sujetos_completos = conteo_mediciones[conteo_mediciones == 4].index.tolist()
    df = df[df['Subject'].isin(sujetos_completos)].copy()
    
    print(f"Sujetos analizados con Cross-Over completo (N = {len(sujetos_completos)}): {sorted(sujetos_completos, key=int)}")
    
    df['Condicion_Cruzada'] = df['Stimulation'] + "_" + df['Time']
    tabla_normalidad, tabla_anova, tabla_posthoc = [], [], []
    
    for metrica, (titulo, ylabel) in config_metricas.items():
        if metrica not in df.columns:
            continue
            
        try:
            col_log = f"log_{metrica}"
            df[col_log] = np.log10(df[metrica])
            col_analisis = col_log if usar_log else metrica
            ylabel_plot = f"Log10 {ylabel}" if usar_log else ylabel
            
            # 1. Graficar Boxplot
            if plot_validation:
                graficar_boxplot(df, col_analisis, titulo, ylabel_plot)
            
            # 2. Pruebas de Normalidad por Condicion_Cruzada
            pvals_raw = pg.normality(data=df, dv=metrica, group='Condicion_Cruzada')['pval'].values
            pvals_log = pg.normality(data=df, dv=col_log, group='Condicion_Cruzada')['pval'].values
            
            tabla_normalidad.append({
                'Metric': metrica,
                'Raw Normal?': all(p > 0.05 for p in pvals_raw),
                'Raw Min p-val': round(min(pvals_raw), 3),
                'Log Normal?': all(p > 0.05 for p in pvals_log),
                'Log Min p-val': round(min(pvals_log), 3)
            })
            
            # 3. rmANOVA de 2 Vías
            anova = pg.rm_anova(data=df, dv=col_analisis, within=['Stimulation', 'Time'], subject='Subject', detailed=True)
            ef_time = anova[anova['Source'] == 'Time']
            ef_stim = anova[anova['Source'] == 'Stimulation']
            ef_int = anova[anova['Source'] == 'Stimulation * Time']
            
            tabla_anova.append({
                'Metric': metrica,
                'Time (F, p)': f"{ef_time['F'].values[0]:.3f}, {ef_time['p-unc'].values[0]:.3f}",
                'Stim (F, p)': f"{ef_stim['F'].values[0]:.3f}, {ef_stim['p-unc'].values[0]:.3f}",
                'Interaction (F, p)': f"{ef_int['F'].values[0]:.3f}, {ef_int['p-unc'].values[0]:.3f}"
            })
            
            # 4. Post-Hoc (Bonferroni)
            ph = pg.pairwise_tests(data=df, dv=col_analisis, within=['Stimulation', 'Time'], subject='Subject', padjust='bonf', parametric=True)
            ph_stim = ph[(ph['Contrast'] == 'Stimulation * Time') & (ph['Stimulation'] == 'STIM')]
            
            for _, row in ph_stim.iterrows():
                tabla_posthoc.append({
                    'Metric': metrica,
                    'Comparison': f"{row['A']} vs {row['B']} (STIM)",
                    'p-corr': round(row['p-corr'], 3),
                    'Hedges g': round(row['hedges'], 3)
                })
                
        except Exception as e:
            print(f"  [!] Error procesando {metrica}: {e}")

    print("\n--- NORMALITY TEST ---")
    print(pd.DataFrame(tabla_normalidad).to_string(index=False))
    print("\n--- rmANOVA RESULTS ---")
    print(pd.DataFrame(tabla_anova).to_string(index=False))
    print("\n--- POST-HOC (STIM CONDITION) ---")
    if tabla_posthoc:
        print(pd.DataFrame(tabla_posthoc).to_string(index=False))
    print("=" * 90)

if __name__ == '__main__':
    ARCHIVO_CSV = "Datos_Clinicos_MVC_9HTP.csv"
    configuracion_metricas = {
        'MVC': ('MVC', 'MVC (Kg)'),
        '9HPT_Derecha': ('9HPT Right', '9HPT (s)'),
        '9HPT_Izquierda': ('9HPT Left', '9HPT (s)')
    }

    generate_validation = False  # Cambiar a True si se desean las gráficas de validación

    run_statistical_pipeline(ARCHIVO_CSV, configuracion_metricas, usar_log=False, plot_validation=generate_validation)

## 1.2) Resting State EEG

In [ ]:
import mne
import numpy as np
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt

# ==============================================================================
# 0. Función de Generación de Gráficas de Validación (PSD Basal)
# ==============================================================================
def generate_validation_plots(epochs, subject_id, condition, time_blk, output_dir):
    """
    Genera gráficas de PSD (4 a 40 Hz) en C3 y C4 para validar visualmente 
    los bloques basales en reposo, sombreando las bandas Alfa y Beta.
    """
    fig_dir = os.path.join(output_dir, "Validation_Plots_Resting")
    os.makedirs(fig_dir, exist_ok=True)

    ch_plot = ['C3', 'C4']
    picks = [ch for ch in ch_plot if ch in epochs.ch_names]
    if not picks:
        return

    fig_psd, axes_psd = plt.subplots(1, len(picks), figsize=(6 * len(picks), 4.5))
    if len(picks) == 1:
        axes_psd = [axes_psd]

    psd_basal = epochs.compute_psd(method="multitaper", fmin=4, fmax=40, picks=picks, verbose=False)
    freqs = psd_basal.freqs

    for i, ch in enumerate(picks):
        ax = axes_psd[i]
        ax.plot(freqs, psd_basal.get_data(picks=ch).mean(axis=0)[0] * 1e12, label=f'Basal ({time_blk})', color='navy')
        ax.axvspan(8, 13, color='yellow', alpha=0.2, label='Alpha (8-13 Hz)')
        ax.axvspan(15, 29, color='pink', alpha=0.3, label='Beta (15-29 Hz)')

        ax.set_title(f"Resting PSD - {ch} ({condition} - {time_blk})", fontweight='bold')
        ax.set_xlabel("Frequency (Hz)")
        ax.set_ylabel("PSD (µV²/Hz)")
        ax.set_yscale('log')
        ax.grid(True)
        if i == 0:
            ax.legend()

    plt.tight_layout()
    fig_psd.savefig(os.path.join(fig_dir, f"RestingPSD_sub-{subject_id}_{condition}_{time_blk}.png"), dpi=300)
    plt.close(fig_psd)

# ==============================================================================
# 1. Función de Extracción Matemática (Potencia Beta por Archivo)
# ==============================================================================
def extract_metrics(epochs):
    """
    Extrae la potencia Beta estática (15 - 29 Hz) promediada a través de las épocas 
    y frecuencias para todos los canales, retornando un DataFrame de 1 fila.
    """
    canales = epochs.ch_names
    
    # Cálculo homogéneo con el bloque 2.3 (Multitaper 15-29 Hz, promediando épocas y frecuencias)
    psd_beta = epochs.compute_psd(method="multitaper", fmin=15, fmax=29, picks=canales, verbose=False).get_data().mean(axis=(0, 2))
    
    dict_beta = {ch: val for ch, val in zip(canales, psd_beta)}
    return pd.DataFrame([dict_beta])

# ==============================================================================
# 2. Pipeline Principal (Con Filtro de Sujetos)
# ==============================================================================
def process_metric(preprocessed_folder, output_csv, sujetos_a_procesar=None, plot_validation=False):
    print(f"Buscando archivos basales en: {preprocessed_folder}")
    
    search_pattern = os.path.join(preprocessed_folder, "*_blk-basal_*_eeg-epo.fif")
    archivos_basales = glob.glob(search_pattern)
    
    if not archivos_basales:
        print("No se encontraron archivos basales. Verifica la ruta.")
        return
        
    print(f"Se encontraron {len(archivos_basales)} archivos. Iniciando extracción...\n")
    lista_resultados = []
    base_out_dir = os.path.dirname(os.path.abspath(output_csv)) or os.getcwd()
    
    for filepath in archivos_basales:
        filename = os.path.basename(filepath)
        
        try:
            partes = filename.split('_')
            subject_id = partes[0].split('-')[1]
            
            if sujetos_a_procesar is not None and subject_id not in sujetos_a_procesar:
                continue
                
            condition = partes[1].split('-')[1].upper()
            
            if 'pre' in filename.lower(): time_blk = 'PRE'
            elif 'post' in filename.lower(): time_blk = 'POST'
            else: continue
                
            print(f"Procesando: Sujeto {subject_id} | {condition} | {time_blk}")
            
            epochs = mne.read_epochs(filepath, preload=True, verbose=False)
            
            if plot_validation:
                generate_validation_plots(epochs, subject_id, condition, time_blk, base_out_dir)
            
            df_resumen = extract_metrics(epochs)
            df_resumen['Subject'] = subject_id
            df_resumen['Stimulation'] = condition
            df_resumen['Time'] = time_blk
            
            lista_resultados.append(df_resumen)
            
        except Exception as e:
            print(f"  [!] Error procesando {filename}: {e}")
            
    print("\nConsolidando tabla maestra...")
    if lista_resultados:
        df_maestro = pd.concat(lista_resultados, ignore_index=True)
        cols_metadatos = ['Subject', 'Stimulation', 'Time']
        cols_canales = [c for c in df_maestro.columns if c not in cols_metadatos]
        df_maestro = df_maestro[cols_metadatos + cols_canales]
        
        df_maestro.to_csv(output_csv, index=False)
        print(f"\n¡Éxito! Matriz estadística exportada a: {output_csv}")
        print(df_maestro.head())

# ==============================================================================
# EJECUCIÓN
# ==============================================================================
if __name__ == '__main__':
    Preprocessed_Data_Folder = r"E:\Yoyi\Preprocessed Data"
    Archivo_Salida_CSV = "Beta_Power_RM_ANOVA.csv"

    generate_validation = False  # Cambiar a True si se desean las gráficas de validación
    
    process_metric(Preprocessed_Data_Folder, Archivo_Salida_CSV, plot_validation=generate_validation)

### Statistical Analysis

In [ ]:
import pandas as pd
import pingouin as pg
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ==============================================================================
# FUNCIÓN DE GRAFICADO (2 NIVELES TEMPORALES: PRE vs POST)
# ==============================================================================
def graficar_boxplot(df, metrica, titulo, ylabel):
    fig = plt.figure(figsize=(6, 4))
    
    ax = sns.boxplot(
        data=df, x='Stimulation', y=metrica, hue='Time',
        order=['STIM', 'SHAM'], hue_order=['PRE', 'POST'],
        color='white', width=0.5, gap=0.15, linewidth=1.5,
        fliersize=6, medianprops=dict(linewidth=2.5)
    )
    
    for patch in ax.patches:
        patch.set_edgecolor('black')
        patch.set_facecolor('white')

    lineas_medianas = [line for line in ax.lines if line.get_linewidth() == 2.5]
    medianas_ordenadas = sorted(lineas_medianas, key=lambda l: sum(l.get_xdata()) / 2)
    colores_estandar = ['#0072B2', '#D55E00', '#0072B2', '#D55E00']
    
    for mediana, color in zip(medianas_ordenadas, colores_estandar):
        mediana.set_color(color)

    plt.title(titulo, fontsize=18, fontweight='bold')
    plt.ylabel(ylabel, fontsize=14, fontweight='bold')
    plt.xlabel('')
    plt.xticks(fontsize=12, fontweight='bold')
    
    if ax.legend_:
        ax.legend_.remove()
        
    y_min, y_max = df[metrica].min(), df[metrica].max()
    rango = y_max - y_min
    offset_y = rango * 0.08
    
    cajas = ax.patches[:4]
    cajas_ordenadas = sorted(cajas, key=lambda p: p.get_path().get_extents().bounds[0])
    posiciones_x = [p.get_path().get_extents().bounds[0] + (p.get_path().get_extents().bounds[2] / 2) for p in cajas_ordenadas]
    etiquetas = ['pre', 'post', 'pre', 'post']
    
    for x, etiqueta, color_txt in zip(posiciones_x, etiquetas, colores_estandar):
        ax.text(x, y_min - offset_y, etiqueta, ha='center', va='top', fontweight='bold', fontsize=11, color=color_txt)
    
    plt.ylim(y_min - (rango * 0.15), y_max + (rango * 0.05))
    plt.tight_layout()
    
    output_dir = os.path.join(os.getcwd(), "output")
    os.makedirs(output_dir, exist_ok=True)
    fig.savefig(os.path.join(output_dir, f"Resting_{metrica}.png"), dpi=600, bbox_inches='tight')
    plt.show()
    plt.close(fig)

# ==============================================================================
# ANÁLISIS ESTADÍSTICO UNIFICADO
# ==============================================================================
def run_statistical_pipeline(csv_filepath, canales, plot_validation=False):
    print("=" * 90)
    print(f"CARGANDO DATOS DESDE: {csv_filepath} | Bloque: Resting State EEG")
    print("=" * 90)
    
    try:
        df = pd.read_csv(csv_filepath)
    except FileNotFoundError:
        print(f"Error: Archivo '{csv_filepath}' no encontrado.")
        return
        
    df['Subject'] = df['Subject'].astype(str)
    df['Stimulation'] = df['Stimulation'].astype(str)
    df['Time'] = df['Time'].astype(str)
    
    sujetos_excluidos_manual = ['1', '2', '10']
    df = df[~df['Subject'].isin(sujetos_excluidos_manual)]

    conteo_mediciones = df.groupby('Subject').size()
    sujetos_completos = conteo_mediciones[conteo_mediciones == 4].index.tolist()
    df = df[df['Subject'].isin(sujetos_completos)].copy()
    
    print(f"Sujetos analizados con Cross-Over completo (N = {len(sujetos_completos)}): {sorted(sujetos_completos, key=int)}")
    
    df['Condicion_Cruzada'] = df['Stimulation'] + "_" + df['Time']
    tabla_normalidad, tabla_anova, tabla_posthoc = [], [], []
    
    for canal in canales:
        if canal not in df.columns:
            continue
            
        try:
            # 1. Transformar a Log10 ANTES de graficar (homologado con 2.3)
            col_analisis = f"log_{canal}"
            df[col_analisis] = np.log10(df[canal])
            
            # 2. Generar Boxplot sobre los datos transformados
            if plot_validation:
                graficar_boxplot(df, col_analisis, f"Resting Beta Power - {canal}", f"Log10 Beta Power - {canal}")
            
            # 3. Pruebas de Normalidad
            pvals_raw = pg.normality(data=df, dv=canal, group='Condicion_Cruzada')['pval'].values
            pvals_log = pg.normality(data=df, dv=col_analisis, group='Condicion_Cruzada')['pval'].values
            
            tabla_normalidad.append({
                'Channel': canal,
                'Raw Normal?': all(p > 0.05 for p in pvals_raw),
                'Raw Min p-val': round(min(pvals_raw), 3),
                'Log Normal?': all(p > 0.05 for p in pvals_log),
                'Log Min p-val': round(min(pvals_log), 3)
            })
            
            # 4. rmANOVA
            anova = pg.rm_anova(data=df, dv=col_analisis, within=['Stimulation', 'Time'], subject='Subject', detailed=True)
            ef_time = anova[anova['Source'] == 'Time']
            ef_stim = anova[anova['Source'] == 'Stimulation']
            ef_int = anova[anova['Source'] == 'Stimulation * Time']
            
            tabla_anova.append({
                'Channel': canal,
                'Time (F, p)': f"{ef_time['F'].values[0]:.3f}, {ef_time['p-unc'].values[0]:.3f}",
                'Stim (F, p)': f"{ef_stim['F'].values[0]:.3f}, {ef_stim['p-unc'].values[0]:.3f}",
                'Interaction (F, p)': f"{ef_int['F'].values[0]:.3f}, {ef_int['p-unc'].values[0]:.3f}"
            })
            
            # 5. Post-Hoc
            ph = pg.pairwise_tests(data=df, dv=col_analisis, within=['Stimulation', 'Time'], subject='Subject', padjust='bonf', parametric=True)
            ph_stim = ph[(ph['Contrast'] == 'Stimulation * Time') & (ph['Stimulation'] == 'STIM')]
            
            for _, row in ph_stim.iterrows():
                tabla_posthoc.append({
                    'Channel': canal,
                    'Comparison': f"{row['A']} vs {row['B']} (STIM)",
                    'p-corr': round(row['p-corr'], 3),
                    'Hedges g': round(row['hedges'], 3)
                })
                
        except Exception as e:
            print(f"  [!] Error procesando el canal {canal}: {e}")

    print("\n--- NORMALITY TEST ---")
    print(pd.DataFrame(tabla_normalidad).to_string(index=False))
    print("\n--- rmANOVA RESULTS ---")
    print(pd.DataFrame(tabla_anova).to_string(index=False))
    print("\n--- POST-HOC (STIM CONDITION) ---")
    if tabla_posthoc:
        print(pd.DataFrame(tabla_posthoc).to_string(index=False))
    print("=" * 90)

if __name__ == '__main__':
    ARCHIVO_CSV = "Beta_Power_RM_ANOVA.csv"
    canales_eeg = ['C3', 'C1', 'Cz', 'C2', 'C4', 'CP1', 'CP2', 'Pz']

    generate_validation = False  # Cambiar a True si se desean las gráficas de validación

    run_statistical_pipeline(ARCHIVO_CSV, canales_eeg, plot_validation=generate_validation)

# 2) Intervention Block Analysis

## 2.1) Kinematic

In [ ]:
import mne
import numpy as np
import pandas as pd
import os
import glob
import math
import matplotlib.pyplot as plt

# Coordenadas ideales y diccionario de eventos CORREGIDO (1: Flexion, 2: Extension)
EndPoints = {
    ("Flexion", "FORWARD"): (-80, 0), ("Flexion", "BACKWARD"): (0, 0),
    ("Extension", "FORWARD"): (80, 0), ("Extension", "BACKWARD"): (0, 0),
    ("Radial D.", "FORWARD"): (0, 70), ("Radial D.", "BACKWARD"): (0, 0),
    ("Ulnar D.", "FORWARD"): (0, -70), ("Ulnar D.", "BACKWARD"): (0, 0),
    ("D1", "FORWARD"): (-57, 50), ("D1", "BACKWARD"): (0, 0),
    ("D2", "FORWARD"): (57, 50), ("D2", "BACKWARD"): (0, 0),
    ("D3", "FORWARD"): (57, -50), ("D3", "BACKWARD"): (0, 0),
    ("D4", "FORWARD"): (-57, -50), ("D4", "BACKWARD"): (0, 0)
}

TargetIDs = {1: "Flexion", 2: "Extension", 3: "Radial D.", 4: "Ulnar D.", 5: "D1", 6: "D2", 7: "D3", 8: "D4"}
FS = 2400

def ComputeAccuracy(movement, RawData, movementDirection):
    Xf, Yf = EndPoints[(movement, movementDirection)]
    idx = int(3/4 * np.shape(RawData)[1]) - 1 if movementDirection == "FORWARD" else np.shape(RawData)[1] - 1
    Xi, Yi = RawData[0, idx], RawData[1, idx]
    return math.sqrt((Xi - Xf)**2 + (Yi - Yf)**2)

def ComputeExcecutionTime(movement, RawData, movementDirection):
    Xf, Yf = EndPoints[(movement, movementDirection)]
    if movementDirection == "FORWARD":
        indexI, indexF, timeOffset = int(1/2 * np.shape(RawData)[1]) - 1, int(3/4 * np.shape(RawData)[1]) - 1, 5.0
    else:
        indexI, indexF, timeOffset = int(3/4 * np.shape(RawData)[1]) - 1, np.shape(RawData)[1] - 1, 7.5

    targetHit = False
    for i in range(indexI, indexF):
        distance = math.sqrt((RawData[0, i] - Xf)**2 + (RawData[1, i] - Yf)**2)
        if distance <= 8:
            if not targetHit:
                ExcecutionTime = (i / FS) - timeOffset
                targetHit = True
        else:
            targetHit = False
    return ExcecutionTime if targetHit else np.nan

def ComputeTrayectoryEfficiency(movement, RawData, movementDirection):
    if movementDirection == "FORWARD":
        Current_Trayectory = RawData[:, int(1/2 * np.shape(RawData)[1]):int(3/4 * np.shape(RawData)[1])]
    else:
        Current_Trayectory = RawData[:, int(3/4 * np.shape(RawData)[1]):np.shape(RawData)[1]]

    IdealTrayectoryDistance = math.sqrt(
        (Current_Trayectory[0, -1] - Current_Trayectory[0, 0])**2 +
        (Current_Trayectory[1, -1] - Current_Trayectory[1, 0])**2
    )
    if IdealTrayectoryDistance == 0:
        return np.nan

    diffs = np.diff(Current_Trayectory[:2, :], axis=1)
    PartialDistanceTraveled = np.sum(np.sqrt(np.sum(diffs**2, axis=0)))
    return PartialDistanceTraveled / IdealTrayectoryDistance

def CheckStarAndEndPoints(movement, RawData, movementDirection):
    XfIdeal, YfIdeal = EndPoints[(movement, movementDirection)]
    if movementDirection == "FORWARD":
        Xi, Yi = RawData[0, int(1/2 * np.shape(RawData)[1])], RawData[1, int(1/2 * np.shape(RawData)[1])]
        Xf, Yf = RawData[0, int(3/4 * np.shape(RawData)[1]) - 1], RawData[1, int(3/4 * np.shape(RawData)[1]) - 1]
        XiIdeal, YiIdeal = EndPoints[(movement, "BACKWARD")]
    else:
        Xi, Yi = RawData[0, int(3/4 * np.shape(RawData)[1])], RawData[1, int(3/4 * np.shape(RawData)[1])]
        Xf, Yf = RawData[0, -1], RawData[1, -1]
        XiIdeal, YiIdeal = EndPoints[(movement, "FORWARD")]

    Initial_Displacement = math.sqrt((Xi - XiIdeal)**2 + (Yi - YiIdeal)**2)
    Final_Displacement = math.sqrt((Xf - XfIdeal)**2 + (Yf - YfIdeal)**2)
    return 1 if (Initial_Displacement < 8 and Final_Displacement < 8) else 0

# ==============================================================================
# 0. Función de Generación de Gráficas de Validación (Trayectorias 2D)
# ==============================================================================
def generate_validation_plots(epochs, subject_id, condition, time_blk, output_dir, run_name=""):
    """
    Grafica las trayectorias 2D (X vs Y) de todos los ensayos del archivo para 
    validar visualmente el desempeño espacial hacia las 8 metas.
    """
    fig_dir = os.path.join(output_dir, "Validation_Plots_Kinematics")
    os.makedirs(fig_dir, exist_ok=True)
    
    datos = epochs.get_data()
    n_pts = datos.shape[2]
    idx_start_fwd = int(0.5 * n_pts)
    idx_end_fwd = int(0.75 * n_pts)
    
    fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
    
    for trial in datos:
        axs[0].plot(trial[0, idx_start_fwd:idx_end_fwd], trial[1, idx_start_fwd:idx_end_fwd], alpha=0.35, lw=1)
        axs[1].plot(trial[0, idx_end_fwd:], trial[1, idx_end_fwd:], alpha=0.35, lw=1)
        
    for ax, direccion in zip(axs, ["FORWARD", "BACKWARD"]):
        for mov, dir_key in EndPoints.keys():
            if dir_key == "FORWARD":
                tx, ty = EndPoints[(mov, "FORWARD")]
                ax.scatter(tx, ty, c='red', s=40, zorder=5)
        ax.scatter(0, 0, c='black', s=50, zorder=5)
        ax.set_title(f"{direccion} ({condition} - {time_blk})", fontweight='bold')
        ax.set_xlabel("X Position")
        ax.set_ylabel("Y Position")
        ax.set_xlim(-100, 100)
        ax.set_ylim(-100, 100)
        ax.grid(True, alpha=0.3)
        
    plt.tight_layout()
    tag = run_name.replace('.fif', '') if run_name else f"sub-{subject_id}_{condition}_{time_blk}"
    fig.savefig(os.path.join(fig_dir, f"Kin_{tag}.png"), dpi=300)
    plt.close(fig)

# ==============================================================================
# 1. Función de Extracción Matemática por Archivo (1 RUN)
# ==============================================================================
def extract_metrics(epochs):
    """
    Extrae las métricas cinemáticas de las fases FORWARD y BACKWARD para cada trial
    y retorna el promedio del run en un DataFrame de 1 fila.
    """
    events = epochs.events[:, 2]
    datos_cinematicos = epochs.get_data()
    lista_metricas_trials = []
    
    for trial_idx, trial_data in enumerate(datos_cinematicos):
        mov_name = TargetIDs[events[trial_idx]]
        
        # --- FORWARD ---
        valid_fwd = CheckStarAndEndPoints(mov_name, trial_data, "FORWARD")
        if valid_fwd:
            prec_fwd = ComputeAccuracy(mov_name, trial_data, "FORWARD")
            time_fwd = ComputeExcecutionTime(mov_name, trial_data, "FORWARD")
            eff_fwd  = ComputeTrayectoryEfficiency(mov_name, trial_data, "FORWARD")
        else:
            prec_fwd = time_fwd = eff_fwd = np.nan
            
        # --- BACKWARD ---
        valid_bwd = CheckStarAndEndPoints(mov_name, trial_data, "BACKWARD")
        if valid_bwd:
            prec_bwd = ComputeAccuracy(mov_name, trial_data, "BACKWARD")
            time_bwd = ComputeExcecutionTime(mov_name, trial_data, "BACKWARD")
            eff_bwd  = ComputeTrayectoryEfficiency(mov_name, trial_data, "BACKWARD")
        else:
            prec_bwd = time_bwd = eff_bwd = np.nan

        lista_metricas_trials.append({
            'Fwd_Valid': valid_fwd, 'Fwd_Precision': prec_fwd,
            'Fwd_ExecTime': time_fwd, 'Fwd_Efficiency': eff_fwd,
            'Bwd_Valid': valid_bwd, 'Bwd_Precision': prec_bwd,
            'Bwd_ExecTime': time_bwd, 'Bwd_Efficiency': eff_bwd
        })
        
    df_trials = pd.DataFrame(lista_metricas_trials)
    return df_trials.mean(numeric_only=True).to_frame().T

# ==============================================================================
# 2. Pipeline Principal (Con Filtro de Sujetos)
# ==============================================================================
def process_metric(preprocessed_folder, output_csv, sujetos_a_procesar=None, plot_validation=False):
    print(f"Buscando archivos cinemáticos en: {preprocessed_folder}")
    
    todos_archivos = glob.glob(os.path.join(preprocessed_folder, "*_kin-epo.fif"))
    archivos_kin = [
        f for f in todos_archivos 
        if ("blk-pre" in os.path.basename(f).lower() or "blk-post" in os.path.basename(f).lower())
    ]
    
    if not archivos_kin:
        print("No se encontraron archivos de cinemática válidos (PRE o POST). Verifica la ruta.")
        return
        
    print(f"Se filtraron {len(archivos_kin)} archivos de tarea motora. Iniciando extracción...\n")
    lista_resultados_runs = []
    base_out_dir = os.path.dirname(os.path.abspath(output_csv)) or os.getcwd()
    
    for filepath in archivos_kin:
        filename = os.path.basename(filepath)
        
        try:
            partes = filename.split('_')
            subject_id = partes[0].split('-')[1]
            
            if sujetos_a_procesar is not None and subject_id not in sujetos_a_procesar:
                continue
                
            condition = partes[1].split('-')[1].upper()
            
            if 'pre' in filename.lower(): time_blk = 'PRE'
            elif 'post' in filename.lower(): time_blk = 'POST'
            else: continue
                
            print(f"Procesando: Sujeto {subject_id} | {condition} | {time_blk} | Archivo: {filename}")
            
            epochs = mne.read_epochs(filepath, preload=True, verbose=False)
            
            if plot_validation:
                generate_validation_plots(epochs, subject_id, condition, time_blk, base_out_dir, run_name=filename)
                
            df_resumen_run = extract_metrics(epochs)
            df_resumen_run['Subject'] = subject_id
            df_resumen_run['Stimulation'] = condition
            df_resumen_run['Time'] = time_blk
            
            lista_resultados_runs.append(df_resumen_run)
            
        except Exception as e:
            print(f"  [!] Error procesando {filename}: {e}")

    print("\nConsolidando tabla maestra y promediando por Sujeto, Estimulación y Fase...")
    if lista_resultados_runs:
        df_todos_runs = pd.concat(lista_resultados_runs, ignore_index=True)
        df_maestro = df_todos_runs.groupby(['Subject', 'Stimulation', 'Time']).mean(numeric_only=True).reset_index()
        
        cols_metadatos = ['Subject', 'Stimulation', 'Time']
        cols_metricas = [c for c in df_maestro.columns if c not in cols_metadatos]
        df_maestro = df_maestro[cols_metadatos + cols_metricas]
        
        df_maestro.to_csv(output_csv, index=False)
        print(f"\n¡Éxito! Matriz cinemática PROMEDIADA exportada a: {output_csv}")
        print(df_maestro.head())

if __name__ == '__main__':
    Preprocessed_Data_Folder = r"E:\Yoyi\Preprocessed Data"
    Archivo_Salida_CSV = "Kinematics_RM_ANOVA.csv"

    generate_validation = False  # Cambiar a True si se desean las gráficas de validación
    
    process_metric(Preprocessed_Data_Folder, Archivo_Salida_CSV, plot_validation=generate_validation)

### Statistical Analysis

In [ ]:
import pandas as pd
import pingouin as pg
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ==============================================================================
# FUNCIÓN DE GRAFICADO (2 NIVELES TEMPORALES: PRE vs POST)
# ==============================================================================
def graficar_boxplot(df, metrica, titulo, ylabel):
    fig = plt.figure(figsize=(6, 4))
    
    ax = sns.boxplot(
        data=df, x='Stimulation', y=metrica, hue='Time',
        order=['STIM', 'SHAM'], hue_order=['PRE', 'POST'],
        color='white', width=0.5, gap=0.15, linewidth=1.5,
        fliersize=6, medianprops=dict(linewidth=2.5)
    )
    
    for patch in ax.patches:
        patch.set_edgecolor('black')
        patch.set_facecolor('white')

    lineas_medianas = [line for line in ax.lines if line.get_linewidth() == 2.5]
    medianas_ordenadas = sorted(lineas_medianas, key=lambda l: sum(l.get_xdata()) / 2)
    colores_estandar = ['#0072B2', '#D55E00', '#0072B2', '#D55E00']
    
    for mediana, color in zip(medianas_ordenadas, colores_estandar):
        mediana.set_color(color)

    plt.title(titulo, fontsize=18, fontweight='bold')
    plt.ylabel(ylabel, fontsize=14, fontweight='bold')
    plt.xlabel('')
    plt.xticks(fontsize=12, fontweight='bold')
    
    if ax.legend_:
        ax.legend_.remove()
        
    y_min, y_max = df[metrica].min(), df[metrica].max()
    rango = y_max - y_min
    offset_y = rango * 0.08
    
    cajas = ax.patches[:4]
    cajas_ordenadas = sorted(cajas, key=lambda p: p.get_path().get_extents().bounds[0])
    posiciones_x = [p.get_path().get_extents().bounds[0] + (p.get_path().get_extents().bounds[2] / 2) for p in cajas_ordenadas]
    etiquetas = ['pre', 'post', 'pre', 'post']
    
    for x, etiqueta, color_txt in zip(posiciones_x, etiquetas, colores_estandar):
        ax.text(x, y_min - offset_y, etiqueta, ha='center', va='top', fontweight='bold', fontsize=11, color=color_txt)
    
    plt.ylim(y_min - (rango * 0.15), y_max + (rango * 0.05))
    plt.tight_layout()
    
    output_dir = os.path.join(os.getcwd(), "output")
    os.makedirs(output_dir, exist_ok=True)
    fig.savefig(os.path.join(output_dir, f"{metrica}.png"), dpi=600, bbox_inches='tight')
    plt.show()
    plt.close(fig)

# ==============================================================================
# ANÁLISIS ESTADÍSTICO UNIFICADO
# ==============================================================================
def run_statistical_pipeline(csv_filepath, config_metricas, usar_log=True, plot_validation=False):
    print("=" * 90)
    print(f"CARGANDO DATOS DESDE: {csv_filepath} | Bloque: Kinematics")
    print("=" * 90)
    
    try:
        df = pd.read_csv(csv_filepath)
    except FileNotFoundError:
        print(f"Error: Archivo '{csv_filepath}' no encontrado.")
        return
        
    df['Subject'] = df['Subject'].astype(str)
    df['Stimulation'] = df['Stimulation'].astype(str)
    df['Time'] = df['Time'].astype(str)
    
    sujetos_excluidos_manual = ['1', '2', '10']
    df = df[~df['Subject'].isin(sujetos_excluidos_manual)]
    
    conteo_mediciones = df.groupby('Subject').size()
    sujetos_completos = conteo_mediciones[conteo_mediciones == 4].index.tolist()
    df = df[df['Subject'].isin(sujetos_completos)].copy()
    
    print(f"Sujetos analizados con Cross-Over completo (N = {len(sujetos_completos)}): {sorted(sujetos_completos, key=int)}")
    
    df['Condicion_Cruzada'] = df['Stimulation'] + "_" + df['Time']
    tabla_normalidad, tabla_anova, tabla_posthoc = [], [], []
    
    for metrica, (titulo, ylabel) in config_metricas.items():
        if metrica not in df.columns:
            continue
            
        try:
            col_log = f"log_{metrica}"
            df[col_log] = np.log10(df[metrica])
            col_analisis = col_log if usar_log else metrica
            ylabel_plot = f"Log10 {ylabel}" if usar_log else ylabel
            
            # 1. Graficar Boxplot
            if plot_validation:
                graficar_boxplot(df, col_analisis, titulo, ylabel_plot)
            
            # 2. Pruebas de Normalidad
            pvals_raw = pg.normality(data=df, dv=metrica, group='Condicion_Cruzada')['pval'].values
            pvals_log = pg.normality(data=df, dv=col_log, group='Condicion_Cruzada')['pval'].values
            
            tabla_normalidad.append({
                'Metric': metrica,
                'Raw Normal?': all(p > 0.05 for p in pvals_raw),
                'Raw Min p-val': round(min(pvals_raw), 3),
                'Log Normal?': all(p > 0.05 for p in pvals_log),
                'Log Min p-val': round(min(pvals_log), 3)
            })
            
            # 3. rmANOVA
            anova = pg.rm_anova(data=df, dv=col_analisis, within=['Stimulation', 'Time'], subject='Subject', detailed=True)
            ef_time = anova[anova['Source'] == 'Time']
            ef_stim = anova[anova['Source'] == 'Stimulation']
            ef_int = anova[anova['Source'] == 'Stimulation * Time']
            
            tabla_anova.append({
                'Metric': metrica,
                'Time (F, p)': f"{ef_time['F'].values[0]:.3f}, {ef_time['p-unc'].values[0]:.3f}",
                'Stim (F, p)': f"{ef_stim['F'].values[0]:.3f}, {ef_stim['p-unc'].values[0]:.3f}",
                'Interaction (F, p)': f"{ef_int['F'].values[0]:.3f}, {ef_int['p-unc'].values[0]:.3f}"
            })
            
            # 4. Post-Hoc
            ph = pg.pairwise_tests(data=df, dv=col_analisis, within=['Stimulation', 'Time'], subject='Subject', padjust='bonf', parametric=True)
            ph_stim = ph[(ph['Contrast'] == 'Stimulation * Time') & (ph['Stimulation'] == 'STIM')]
            
            for _, row in ph_stim.iterrows():
                tabla_posthoc.append({
                    'Metric': metrica,
                    'Comparison': f"{row['A']} vs {row['B']} (STIM)",
                    'p-corr': round(row['p-corr'], 3),
                    'Hedges g': round(row['hedges'], 3)
                })
                
        except Exception as e:
            print(f"  [!] Error procesando {metrica}: {e}")

    print("\n--- NORMALITY TEST ---")
    print(pd.DataFrame(tabla_normalidad).to_string(index=False))
    print("\n--- rmANOVA RESULTS ---")
    print(pd.DataFrame(tabla_anova).to_string(index=False))
    print("\n--- POST-HOC (STIM CONDITION) ---")
    if tabla_posthoc:
        print(pd.DataFrame(tabla_posthoc).to_string(index=False))
    print("=" * 90)

if __name__ == '__main__':
    ARCHIVO_CSV = "Kinematics_RM_ANOVA.csv"
    configuracion_metricas = {
        'Fwd_Precision': ('Forward Precision', 'Error'),
        'Fwd_ExecTime': ('Forward Execution Time', 'Time (s)'),
        'Fwd_Efficiency': ('Forward Efficiency', 'Efficiency Ratio'),
        'Bwd_Precision': ('Backward Precision', 'Error'),
        'Bwd_ExecTime': ('Backward Execution Time', 'Time (s)'),
        'Bwd_Efficiency': ('Backward Efficiency', 'Efficiency Ratio')
    }

    generate_validation = False  # Cambiar a True si se desean las gráficas de validación

    run_statistical_pipeline(ARCHIVO_CSV, configuracion_metricas, usar_log=True, plot_validation=generate_validation)

## 2.2) PSD & ERD/ERS

In [ ]:
import mne
import numpy as np
import pandas as pd
import os
import glob
import gc
import matplotlib.pyplot as plt
import scipy.stats as stats

# ==============================================================================
# 0. Función de Generación de Gráficas de Validación
# ==============================================================================
def generate_validation_plots(epochs, subject_id, condition, time_blk, output_dir):
    """
    Genera gráficas de PSD y mapas de tiempo-frecuencia (ERD/ERS) para validar
    los datos, optimizando el uso de memoria RAM con decim=10.
    """
    fig_dir = os.path.join(output_dir, "Validation_Plots")
    os.makedirs(fig_dir, exist_ok=True)

    ch_plot = ['C3', 'C4']
    picks = [ch for ch in ch_plot if ch in epochs.ch_names]
    
    if not picks:
        return

    # ---------------------------------------------------------
    # A. Gráfica de PSD (Power Spectral Density)
    # ---------------------------------------------------------
    fig_psd, axes_psd = plt.subplots(1, len(picks), figsize=(6 * len(picks), 5))
    if len(picks) == 1: axes_psd = [axes_psd]

    t_relax = (3.0, 5.0)
    t_fwd = (5.0, 7.0)
    t_bwd = (7.5, 9.5)

    psd_relax = epochs.copy().crop(*t_relax).compute_psd(method="multitaper", fmin=4, fmax=40, picks=picks, verbose=False)
    psd_fwd = epochs.copy().crop(*t_fwd).compute_psd(method="multitaper", fmin=4, fmax=40, picks=picks, verbose=False)
    psd_bwd = epochs.copy().crop(*t_bwd).compute_psd(method="multitaper", fmin=4, fmax=40, picks=picks, verbose=False)

    freqs_relax = psd_relax.freqs
    freqs_fwd = psd_fwd.freqs
    freqs_bwd = psd_bwd.freqs

    for i, ch in enumerate(picks):
        ax = axes_psd[i]
        ax.plot(freqs_relax, psd_relax.get_data(picks=ch).mean(axis=0)[0] * 1e12, label='Relax (3-5s)', color='blue')
        ax.plot(freqs_fwd, psd_fwd.get_data(picks=ch).mean(axis=0)[0] * 1e12, label='Forward (5-7.0s)', color='red')
        ax.plot(freqs_bwd, psd_bwd.get_data(picks=ch).mean(axis=0)[0] * 1e12, label='Backward (7.5-9.5s)', color='green')

        ax.axvspan(8, 13, color='yellow', alpha=0.2, label='alfa band')
        ax.axvspan(15, 30, color='pink', alpha=0.3, label='beta band')

        ax.set_title(f"PSD - {ch} ({condition} - {time_blk})", fontweight='bold')
        ax.set_xlabel("Frequency (Hz)")
        ax.set_ylabel("Power Spectral Density (µV²/Hz)")
        ax.grid(True)
        if i == 0: ax.legend()

        ax.set_yscale('log')
        ax.autoscale(enable=True, axis='y', tight=False)

    plt.tight_layout()
    fig_psd.savefig(os.path.join(fig_dir, f"PSD_sub-{subject_id}_{condition}_{time_blk}.png"), dpi=300)
    plt.close(fig_psd)
    del psd_relax, psd_fwd, psd_bwd, fig_psd

    # ---------------------------------------------------------
    # B. Mapa Tiempo-Frecuencia (ERD/ERS) CON MÁSCARA ESTADÍSTICA
    # ---------------------------------------------------------
    freqs_tfr = np.arange(1, 41, 1)
    n_cycles = freqs_tfr / 2.0

    epochs_tfr = epochs.copy().crop(3.0, 9.99)
    
    # OPTIMIZACIÓN CLAVE: decim=10 reduce el tamaño en RAM 10 veces (de 1.28 GB a 128 MB)
    power = epochs_tfr.compute_tfr(
        method="morlet", freqs=freqs_tfr, n_cycles=n_cycles,
        return_itc=False, average=False, picks=picks, decim=10, verbose=False
    )

    power.apply_baseline(baseline=(3.0, 5.0), mode='percent', verbose=False)
    t_vals, p_vals = stats.ttest_1samp(power.data, 0, axis=0)
    significance_mask = p_vals < 0.05
    avg_power = power.average()

    # Liberar matriz 4D pesada antes de graficar
    del power, epochs_tfr, t_vals, p_vals

    fig_tfr, axes_tfr = plt.subplots(1, len(picks), figsize=(6 * len(picks), 5))
    if len(picks) == 1: axes_tfr = [axes_tfr]

    for i, ch in enumerate(picks):
        ch_idx = avg_power.ch_names.index(ch)
        ch_mask = significance_mask[ch_idx]

        avg_power.plot(
            picks=[ch], axes=axes_tfr[i], vlim=(-0.5, 0.5),
            cmap='RdBu_r', colorbar=(i == len(picks)-1),
            mask=ch_mask, mask_style='mask', mask_alpha=0.0, show=False
        )
        
        axes_tfr[i].set_title(f"ERD/ERS - {ch} ({condition} - {time_blk})", fontweight='bold')
        axes_tfr[i].axvline(5.0, color='black', linestyle='--', linewidth=1.5)
        axes_tfr[i].axvline(7.5, color='black', linestyle='--', linewidth=1.5)
        axes_tfr[i].set_xlim(3.0, 10.0)

    plt.tight_layout()
    fig_tfr.savefig(os.path.join(fig_dir, f"ERD_sub-{subject_id}_{condition}_{time_blk}.png"), dpi=300)
    plt.close(fig_tfr)
    del avg_power, significance_mask, fig_tfr

# ==============================================================================
# 1. Función de Extracción Matemática para el ANOVA (Completa por Fases)
# ==============================================================================
def extract_task_eeg_metrics(epochs):
    """
    Extrae PSD Beta (15-29 Hz) vía Multitaper y ERD/ERS Beta vía Morlet
    optimizando el consumo de memoria RAM.
    """
    canales = epochs.ch_names
    
    # A. Cálculo del PSD estático (Método Multitaper, 15-29 Hz)
    psd_all   = epochs.compute_psd(method="multitaper", fmin=15, fmax=29, verbose=False).get_data().mean(axis=(0, 2))
    psd_relax = epochs.copy().crop(3.0, 5.0).compute_psd(method="multitaper", fmin=15, fmax=29, verbose=False).get_data().mean(axis=(0, 2))
    psd_fwd   = epochs.copy().crop(5.0, 7.0).compute_psd(method="multitaper", fmin=15, fmax=29, verbose=False).get_data().mean(axis=(0, 2))
    psd_bwd   = epochs.copy().crop(7.5, 9.5).compute_psd(method="multitaper", fmin=15, fmax=29, verbose=False).get_data().mean(axis=(0, 2))
    
    # B. Cálculo del ERD/ERS dinámico (Morlet Wavelets, 15-29 Hz)
    freqs = np.arange(15, 30, 1)
    n_cycles = freqs / 2.0
    
    # Recortamos a (3.0, 9.5) y usamos decim=10 para reducir la matriz de 2.75 GB a ~180 MB
    epochs_crop = epochs.copy().crop(3.0, 9.5)
    power = epochs_crop.compute_tfr(
        method="morlet", freqs=freqs, n_cycles=n_cycles,
        return_itc=False, average=False, decim=10, verbose=False
    )
    
    times = power.times
    datos_beta = power.data
    
    mascara_basal = (times >= 3.0) & (times <= 5.0)
    mascara_fwd   = (times >= 5.0) & (times <= 7.0)
    mascara_bwd   = (times >= 7.5) & (times <= 9.5)
    
    potencia_basal = np.mean(datos_beta[:, :, :, mascara_basal], axis=(2, 3))
    potencia_fwd   = np.mean(datos_beta[:, :, :, mascara_fwd], axis=(2, 3))
    potencia_bwd   = np.mean(datos_beta[:, :, :, mascara_bwd], axis=(2, 3))
    
    erd_fwd_promedio = np.mean(((potencia_fwd - potencia_basal) / potencia_basal) * 100, axis=0)
    erd_bwd_promedio = np.mean(((potencia_bwd - potencia_basal) / potencia_basal) * 100, axis=0)
    
    del power, datos_beta, epochs_crop
    
    # C. Consolidación de Diccionarios para el CSV
    dict_psd_all   = {f"PSD_{ch}": val for ch, val in zip(canales, psd_all)}
    dict_psd_relax = {f"PSD_RELAX_{ch}": val for ch, val in zip(canales, psd_relax)}
    dict_psd_fwd   = {f"PSD_FWD_{ch}": val for ch, val in zip(canales, psd_fwd)}
    dict_psd_bwd   = {f"PSD_BWD_{ch}": val for ch, val in zip(canales, psd_bwd)}
    dict_erd_fwd   = {f"ERD_FWD_{ch}": val for ch, val in zip(canales, erd_fwd_promedio)}
    dict_erd_bwd   = {f"ERD_BWD_{ch}": val for ch, val in zip(canales, erd_bwd_promedio)}
    
    return pd.DataFrame([{
        **dict_psd_all, **dict_psd_relax, **dict_psd_fwd, **dict_psd_bwd,
        **dict_erd_fwd, **dict_erd_bwd
    }])

# ==============================================================================
# 2. Pipeline Principal (Con Limpieza Automática de RAM)
# ==============================================================================
def process_task_eeg(preprocessed_folder, output_csv, sujetos_a_procesar=None, plot_validation=False):
    print(f"Buscando archivos de tarea EEG en: {preprocessed_folder}")
    
    if sujetos_a_procesar:
        print(f"-> FILTRO ACTIVADO: Solo se procesarán los sujetos: {sujetos_a_procesar}")
        
    search_pattern = os.path.join(preprocessed_folder, "*_eeg-epo.fif")
    archivos = [f for f in glob.glob(search_pattern) if 'blk-basal' not in os.path.basename(f)]
    
    if not archivos:
        print("No se encontraron archivos válidos. Verifica la ruta.")
        return
        
    print(f"Se encontraron {len(archivos)} archivos en total. Iniciando extracción...\n")
    lista_resultados = []
    base_out_dir = os.path.dirname(os.path.abspath(output_csv)) or os.getcwd()
    
    for filepath in archivos:
        filename = os.path.basename(filepath)
        epochs = None
        
        try:
            partes = filename.split('_')
            subject_id = partes[0].split('-')[1]
            
            if sujetos_a_procesar is not None and subject_id not in sujetos_a_procesar:
                continue
                
            condition = partes[1].split('-')[1].upper()
            
            if 'pre' in filename.lower(): time_blk = 'PRE'
            elif 'during' in filename.lower(): time_blk = 'DURING'
            elif 'post' in filename.lower(): time_blk = 'POST'
            else: continue
                
            print(f"Procesando: Sujeto {subject_id} | {condition} | {time_blk}")
            
            epochs = mne.read_epochs(filepath, preload=True, verbose=False)
            
            if plot_validation:
                generate_validation_plots(epochs, subject_id, condition, time_blk, base_out_dir)
            
            df_resumen = extract_task_eeg_metrics(epochs)
            df_resumen['Subject'] = subject_id
            df_resumen['Stimulation'] = condition
            df_resumen['Time'] = time_blk
            
            lista_resultados.append(df_resumen)
            
        except Exception as e:
            print(f"  [!] Error procesando {filename}: {e}")
        finally:
            # Limpieza garantizada al final de cada archivo
            if epochs is not None:
                del epochs
            plt.close('all')
            gc.collect()
            
    print("\nConsolidando tabla maestra...")
    if lista_resultados:
        df_maestro = pd.concat(lista_resultados, ignore_index=True)
        
        cols_metadatos = ['Subject', 'Stimulation', 'Time']
        cols_metricas = [c for c in df_maestro.columns if c not in cols_metadatos]
        df_maestro = df_maestro[cols_metadatos + cols_metricas]
        
        df_maestro.to_csv(output_csv, index=False)
        print(f"\n¡Éxito! Matriz estadística exportada a: {output_csv}")
        print("Tus gráficas de validación se guardaron en la carpeta 'Validation_Plots'")
    else:
        print("\n[!] No se generaron resultados para exportar.")

# ==============================================================================
# EJECUCIÓN
# ==============================================================================
if __name__ == '__main__':
    Preprocessed_Data_Folder = r"E:\Yoyi\Preprocessed Data"
    Archivo_Salida_CSV = "TaskEEG_RM_ANOVA.csv"

    generate_validation = False  # Cambiar a True si se desean las gráficas de validación
    
    process_task_eeg(Preprocessed_Data_Folder, Archivo_Salida_CSV, plot_validation=generate_validation)

### Statistical Analysis

In [ ]:
import pandas as pd
import pingouin as pg
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore') 
# ==============================================================================
# FUNCIÓN DE GRAFICADO (3 NIVELES TEMPORALES)
# ==============================================================================
def graficar_boxplot(df, metrica, titulo, ylabel):
        fig = plt.figure(figsize=(8, 5))
        
        ax = sns.boxplot(
            data=df, x='Stimulation', y=metrica, hue='Time', 
            order=['STIM', 'SHAM'], hue_order=['PRE', 'DURING', 'POST'],
            color='white', width=0.6, gap=0.1, linewidth=1.5, 
            fliersize=6, medianprops=dict(linewidth=2.5) 
        )
        
        for patch in ax.patches:
            patch.set_edgecolor('black')
            patch.set_facecolor('white')

        # Ordenar 6 medianas de izquierda a derecha
        lineas_medianas = [line for line in ax.lines if line.get_linewidth() == 2.5]
        medianas_ordenadas = sorted(lineas_medianas, key=lambda l: sum(l.get_xdata()) / 2)
        
        # Colores visuales: [STIM-PRE, DUR, POST, SHAM-PRE, DUR, POST]
        colores_estandar = ['#0072B2', '#E69F00', '#D55E00', '#0072B2', '#E69F00', '#D55E00'] 
        
        for mediana, color in zip(medianas_ordenadas, colores_estandar):
            mediana.set_color(color)

        plt.title(titulo, fontsize=18, fontweight='bold')
        plt.ylabel(ylabel, fontsize=14, fontweight='bold')
        plt.xlabel('') 
        plt.xticks(fontsize=12, fontweight='bold')
        
        if ax.legend_:
            ax.legend_.remove()
            
        y_min, y_max = df[metrica].min(), df[metrica].max()
        rango = y_max - y_min
        offset_y = rango * 0.08 
        
        cajas = ax.patches[:6]
        cajas_ordenadas = sorted(cajas, key=lambda p: p.get_path().get_extents().bounds[0])
        
        posiciones_x = [p.get_path().get_extents().bounds[0] + (p.get_path().get_extents().bounds[2] / 2) for p in cajas_ordenadas]
        etiquetas = ['pre', 'dur', 'post', 'pre', 'dur', 'post']
        
        for x, etiqueta, color_txt in zip(posiciones_x, etiquetas, colores_estandar):
            ax.text(x, y_min - offset_y, etiqueta, ha='center', va='top', fontweight='bold', fontsize=11, color=color_txt)
        
        plt.ylim(y_min - (rango * 0.15), y_max + (rango * 0.05))
        plt.tight_layout()
        
        output_dir = os.path.join(os.getcwd(), "output")
        os.makedirs(output_dir, exist_ok=True)
        fig.savefig(os.path.join(output_dir, f"{metrica}.png"), dpi=600, bbox_inches='tight')
        plt.close()
# ==============================================================================
# ANÁLISIS ESTADÍSTICO
# ==============================================================================
def run_statistical_pipeline(csv_filepath, canales, tipo_metrica="PSD", plot_validation=False):
    print("=" * 90)
    print(f"CARGANDO DATOS DESDE: {csv_filepath} | Métrica: {tipo_metrica}")
    print("=" * 90)
    
    try:
        df = pd.read_csv(csv_filepath)
    except FileNotFoundError:
        print(f"Error: Archivo '{csv_filepath}' no encontrado.")
        return
        
    df['Subject'] = df['Subject'].astype(str)
    df['Stimulation'] = df['Stimulation'].astype(str)
    df['Time'] = df['Time'].astype(str)
    
    sujetos_excluidos_manual = ['1', '2', '10']
    df = df[~df['Subject'].isin(sujetos_excluidos_manual)]
    # Filtro Estricto: Exactamente 6 mediciones (2 condiciones x 3 tiempos)
    conteo_mediciones = df.groupby('Subject').size()
    sujetos_completos = conteo_mediciones[conteo_mediciones == 6].index.tolist()
    df = df[df['Subject'].isin(sujetos_completos)]
    
    print(f"Sujetos analizados con 6 bloques completos (N = {len(sujetos_completos)}): {sorted(sujetos_completos, key=int)}")
    
    df['Condicion_Cruzada'] = df['Stimulation'] + "_" + df['Time']
    
    tabla_normalidad, tabla_anova, tabla_posthoc = [], [], []
    
    for canal in canales:
        metrica_col = f"{tipo_metrica}_{canal}"
        if metrica_col not in df.columns:
            continue
            
        try:
            # 1. Definir la columna de análisis y la transformación ANTES de graficar
            if "ERD" in tipo_metrica:
                col_analisis = metrica_col
                is_log = False
                ylabel_plot = f"ERD/ERS (%) - {canal}"
            else:
                col_analisis = f'log_{metrica_col}'
                df[col_analisis] = np.log10(df[metrica_col])
                is_log = True
                ylabel_plot = f"Log10 Beta Power - {canal}"
                
            # 2. Generar el Boxplot usando los datos transformados (col_analisis)
            if plot_validation:
                graficar_boxplot(df, col_analisis, f"{tipo_metrica} Beta - {canal}", ylabel_plot)
            
            # 3. Pruebas de Normalidad
            pvals_raw = pg.normality(data=df, dv=metrica_col, group='Condicion_Cruzada')['pval'].values
            pvals_analysis = pg.normality(data=df, dv=col_analisis, group='Condicion_Cruzada')['pval'].values
            
            tabla_normalidad.append({
                'Channel': canal, 
                'Raw Normal?': all(p > 0.05 for p in pvals_raw), 
                f'{"Log" if is_log else "Raw"} Min p-val': round(min(pvals_analysis), 3)
            })
            
            # 4. rmANOVA
            anova = pg.rm_anova(data=df, dv=col_analisis, within=['Stimulation', 'Time'], subject='Subject', detailed=True)
            ef_stim = anova[anova['Source'] == 'Stimulation']
            ef_time = anova[anova['Source'] == 'Time']
            ef_int = anova[anova['Source'] == 'Stimulation * Time']
            
            tabla_anova.append({
                'Channel': canal,
                'Time (F, p)': f"{ef_time['F'].values[0]:.3f}, {ef_time['p-unc'].values[0]:.3f}",
                'Stim (F, p)': f"{ef_stim['F'].values[0]:.3f}, {ef_stim['p-unc'].values[0]:.3f}",
                'Interaction (F, p)': f"{ef_int['F'].values[0]:.3f}, {ef_int['p-unc'].values[0]:.3f}"
            })
            
            # 5. Post-Hoc
            ph = pg.pairwise_tests(data=df, dv=col_analisis, within=['Stimulation', 'Time'], subject='Subject', padjust='bonf', parametric=True)
            ph_stim = ph[(ph['Contrast'] == 'Stimulation * Time') & (ph['Stimulation'] == 'STIM')]
            
            for _, row in ph_stim.iterrows():
                if row['p-corr'] < 0.1: # Mostrar tendencias o significativos
                    tabla_posthoc.append({
                        'Channel': canal, 
                        'Comparison': f"{row['A']} vs {row['B']} (STIM)",
                        'p-corr': round(row['p-corr'], 3),
                        'Hedges g': round(row['hedges'], 3)
                    })
                
        except Exception as e:
            print(f"  [!] Error procesando el canal {canal}: {e}")
    print("\n--- NORMALITY TEST ---")
    print(pd.DataFrame(tabla_normalidad).to_string(index=False))
    print("\n--- rmANOVA RESULTS ---")
    print(pd.DataFrame(tabla_anova).to_string(index=False))
    print("\n--- POST-HOC (STIM CONDITION) ---")
    if tabla_posthoc:
        print(pd.DataFrame(tabla_posthoc).to_string(index=False))
    else:
        print("No post-hoc contrasts reached p < 0.1")
if __name__ == '__main__':
    ARCHIVO_CSV = "TaskEEG_RM_ANOVA.csv"
    canales_eeg = ['C3', 'C1', 'Cz', 'C2', 'C4', 'CP1', 'CP2', 'Pz']

    generate_validation = False  # Cambiar a True si se desean las gráficas de validación
    
    # 1. PSD (Global y por fases de 2 segundos)
    run_statistical_pipeline(ARCHIVO_CSV, canales_eeg, tipo_metrica="PSD", plot_validation=generate_validation)
    run_statistical_pipeline(ARCHIVO_CSV, canales_eeg, tipo_metrica="PSD_RELAX", plot_validation=generate_validation)
    run_statistical_pipeline(ARCHIVO_CSV, canales_eeg, tipo_metrica="PSD_FWD", plot_validation=generate_validation)
    run_statistical_pipeline(ARCHIVO_CSV, canales_eeg, tipo_metrica="PSD_BWD", plot_validation=generate_validation)

    # 2. ERD/ERS (Por fases de 2 segundos)
    run_statistical_pipeline(ARCHIVO_CSV, canales_eeg, tipo_metrica="ERD_FWD", plot_validation=generate_validation)
    run_statistical_pipeline(ARCHIVO_CSV, canales_eeg, tipo_metrica="ERD_BWD", plot_validation=generate_validation)

## 2.3) CCI

In [ ]:
import mne
import numpy as np
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import gc

# ==============================================================================
# CONFIGURACIÓN DE PARES MUSCULARES (AGONISTA vs ANTAGONISTA)
# ==============================================================================
# Ajusta los nombres según aparezcan en epochs.ch_names de tus archivos *_emg-epo.fif.
# Si los nombres en el archivo son distintos, el script emparejará automáticamente 
# los canales disponibles (ej. Canal 0 vs Canal 1).
PARES_MUSCULARES = {
    'FlexExt_Global': (0, 1)  # Índices (o nombres) de los canales EMG a comparar
}

def calcular_envolvente_emg(data, sfreq, cutoff_hz=6.0):
    """
    Rectifica la señal EMG (valor absoluto tras quitar offset DC) y aplica un 
    filtro pasa-bajas Butterworth de 4to orden (6 Hz) para obtener la envolvente lineal.
    """
    # Eliminar offset DC por época y canal
    data_zero = data - np.mean(data, axis=-1, keepdims=True)
    # Rectificación de onda completa
    data_rect = np.abs(data_zero)
    # Filtro pasa-bajas para envolvente lineal
    nyq = 0.5 * sfreq
    b, a = butter(4, cutoff_hz / nyq, btype='low')
    envelope = filtfilt(b, a, data_rect, axis=-1)
    # Asegurar valores positivos y normalizar respecto al pico máximo de cada canal en el bloque
    envelope = np.clip(envelope, a_min=0, a_max=None)
    max_vals = np.percentile(envelope, 99, axis=(0, 2), keepdims=True)
    max_vals[max_vals == 0] = 1.0
    return envelope / max_vals

def compute_cci_window(env_ch1, env_ch2):
    """
    Calcula el Índice de Cocontracción (Falconer & Winter) época por época 
    y devuelve el promedio (%) del bloque.
    env_ch1, env_ch2: arrays de shape (n_epochs, n_times_window)
    """
    min_env = np.minimum(env_ch1, env_ch2)
    num = 2.0 * np.trapz(min_env, axis=-1)
    den = np.trapz(env_ch1 + env_ch2, axis=-1)
    den[den == 0] = np.nan
    cci_epochs = (num / den) * 100.0
    return np.nanmean(cci_epochs)

# ==============================================================================
# 0. Función de Generación de Gráficas de Validación (Envolventes y Solapamiento)
# ==============================================================================
def generate_validation_plots(epochs, subject_id, condition, time_blk, output_dir, run_name=""):
    """
    Grafica las envolventes EMG promedio de los músculos agonista y antagonista
    y sombrea el área de cocontracción (solapamiento) a lo largo de la tarea (3-10 s).
    """
    fig_dir = os.path.join(output_dir, "Validation_Plots_CCI")
    os.makedirs(fig_dir, exist_ok=True)
    
    sfreq = epochs.info['sfreq']
    times = epochs.times
    data = epochs.get_data()
    
    if data.shape[1] < 2:
        return
        
    env = calcular_envolvente_emg(data, sfreq)
    # Promediar a través de las épocas para visualización
    env_mean = np.mean(env, axis=0)
    
    mask_t = (times >= 3.0) & (times <= 9.99)
    t_plot = times[mask_t]
    ch1_plot = env_mean[0, mask_t]
    ch2_plot = env_mean[1, mask_t]
    overlap = np.minimum(ch1_plot, ch2_plot)
    
    ch1_name = epochs.ch_names[0]
    ch2_name = epochs.ch_names[1]
    
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(t_plot, ch1_plot, label=f"EMG 1 ({ch1_name})", color='#0072B2', lw=2)
    ax.plot(t_plot, ch2_plot, label=f"EMG 2 ({ch2_name})", color='#D55E00', lw=2)
    ax.fill_between(t_plot, 0, overlap, color='gray', alpha=0.35, label='Co-contraction Area (CCI)')
    
    ax.axvline(5.0, color='black', linestyle='--', linewidth=1.5, label='Forward Onset (5.0s)')
    ax.axvline(7.5, color='black', linestyle=':', linewidth=1.5, label='Backward Onset (7.5s)')
    
    ax.set_title(f"EMG Co-Contraction - Sub {subject_id} ({condition} - {time_blk})", fontweight='bold')
    ax.set_xlabel("Time (s)", fontweight='bold')
    ax.set_ylabel("Normalized EMG Envelope (a.u.)", fontweight='bold')
    ax.set_xlim(3.0, 10.0)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')
    
    plt.tight_layout()
    tag = run_name.replace('.fif', '') if run_name else f"sub-{subject_id}_{condition}_{time_blk}"
    fig.savefig(os.path.join(fig_dir, f"CCI_{tag}.png"), dpi=300)
    plt.close(fig)

# ==============================================================================
# 1. Función de Extracción Matemática (CCI por Fases)
# ==============================================================================
def extract_metrics(epochs):
    """
    Extrae el CCI (%) en ventanas simétricas de 2.0 segundos:
    - CCI_ALL (Movimiento completo: 5.0 a 9.5 s)
    - CCI_RELAX (Preparación/Reposo: 3.0 a 5.0 s)
    - CCI_FWD (Ejecución Ida: 5.0 a 7.0 s)
    - CCI_BWD (Ejecución Regreso: 7.5 a 9.5 s)
    """
    sfreq = epochs.info['sfreq']
    times = epochs.times
    data = epochs.get_data()
    n_ch = data.shape[1]
    
    env = calcular_envolvente_emg(data, sfreq)
    
    mask_all   = (times >= 5.0) & (times <= 9.5)
    mask_relax = (times >= 3.0) & (times <= 5.0)
    mask_fwd   = (times >= 5.0) & (times <= 7.0)
    mask_bwd   = (times >= 7.5) & (times <= 9.5)
    
    resultados = {}
    
    # Evaluar los pares configurados
    for pair_name, (idx1, idx2) in PARES_MUSCULARES.items():
        # Resolver índices si se pasaron nombres de canales
        c1 = epochs.ch_names.index(idx1) if isinstance(idx1, str) and idx1 in epochs.ch_names else (idx1 if isinstance(idx1, int) and idx1 < n_ch else 0)
        c2 = epochs.ch_names.index(idx2) if isinstance(idx2, str) and idx2 in epochs.ch_names else (idx2 if isinstance(idx2, int) and idx2 < n_ch else min(1, n_ch - 1))
        
        resultados[f"CCI_ALL_{pair_name}"]   = compute_cci_window(env[:, c1, mask_all], env[:, c2, mask_all])
        resultados[f"CCI_RELAX_{pair_name}"] = compute_cci_window(env[:, c1, mask_relax], env[:, c2, mask_relax])
        resultados[f"CCI_FWD_{pair_name}"]   = compute_cci_window(env[:, c1, mask_fwd], env[:, c2, mask_fwd])
        resultados[f"CCI_BWD_{pair_name}"]   = compute_cci_window(env[:, c1, mask_bwd], env[:, c2, mask_bwd])
        
    # Si el archivo tiene 4 canales EMG (ej. 2 flexores y 2 extensores), calcular también el par secundario (2 vs 3)
    if n_ch >= 4 and len(PARES_MUSCULARES) == 1:
        resultados["CCI_ALL_Pair2"]   = compute_cci_window(env[:, 2, mask_all], env[:, 3, mask_all])
        resultados["CCI_RELAX_Pair2"] = compute_cci_window(env[:, 2, mask_relax], env[:, 3, mask_relax])
        resultados["CCI_FWD_Pair2"]   = compute_cci_window(env[:, 2, mask_fwd], env[:, 3, mask_fwd])
        resultados["CCI_BWD_Pair2"]   = compute_cci_window(env[:, 2, mask_bwd], env[:, 3, mask_bwd])
        
    return pd.DataFrame([resultados])

# ==============================================================================
# 2. Pipeline Principal (Con Filtro de Sujetos y Promediado de Runs)
# ==============================================================================
def process_metric(preprocessed_folder, output_csv, sujetos_a_procesar=None, plot_validation=False):
    print(f"Buscando archivos EMG de tarea motora en: {preprocessed_folder}")
    
    search_pattern = os.path.join(preprocessed_folder, "*_emg-epo.fif")
    archivos_emg = [f for f in glob.glob(search_pattern) if 'blk-basal' not in os.path.basename(f).lower()]
    
    if not archivos_emg:
        print("No se encontraron archivos *_emg-epo.fif válidos. Verifica la ruta.")
        return
        
    print(f"Se encontraron {len(archivos_emg)} archivos EMG. Iniciando extracción de CCI...\n")
    lista_resultados = []
    base_out_dir = os.path.dirname(os.path.abspath(output_csv)) or os.getcwd()
    
    for idx, filepath in enumerate(archivos_emg):
        filename = os.path.basename(filepath)
        
        try:
            partes = filename.split('_')
            subject_id = partes[0].split('-')[1]
            
            if sujetos_a_procesar is not None and subject_id not in sujetos_a_procesar:
                continue
                
            condition = partes[1].split('-')[1].upper()
            
            if 'pre' in filename.lower(): time_blk = 'PRE'
            elif 'during' in filename.lower(): time_blk = 'DURING'
            elif 'post' in filename.lower(): time_blk = 'POST'
            else: continue
                
            print(f"Procesando CCI: Sujeto {subject_id} | {condition} | {time_blk} | Archivo: {filename}")
            epochs = mne.read_epochs(filepath, preload=True, verbose=False)
            
            if idx == 0:
                print(f"  -> Canales EMG detectados en archivo: {epochs.ch_names}")
                
            if plot_validation:
                generate_validation_plots(epochs, subject_id, condition, time_blk, base_out_dir, run_name=filename)
                
            df_resumen = extract_metrics(epochs)
            df_resumen['Subject'] = subject_id
            df_resumen['Stimulation'] = condition
            df_resumen['Time'] = time_blk
            
            lista_resultados.append(df_resumen)

        # Al final del bloque try...except dentro del bucle for de CCI y CMC:
        except Exception as e:
            print(f"  [!] Error procesando {filename}: {e}")
        finally:
            plt.close('all')
            gc.collect()
            
    print("\nConsolidando tabla maestra y promediando por Sujeto, Estimulación y Fase...")
    if lista_resultados:
        df_todos = pd.concat(lista_resultados, ignore_index=True)
        df_maestro = df_todos.groupby(['Subject', 'Stimulation', 'Time']).mean(numeric_only=True).reset_index()
        
        cols_metadatos = ['Subject', 'Stimulation', 'Time']
        cols_metricas = [c for c in df_maestro.columns if c not in cols_metadatos]
        df_maestro = df_maestro[cols_metadatos + cols_metricas]
        
        df_maestro.to_csv(output_csv, index=False)
        print(f"\n¡Éxito! Matriz de Cocontracción exportada a: {output_csv}")
        print(df_maestro.head())
    else:
        print("\n[!] No se generaron resultados para exportar.")

# ==============================================================================
# EJECUCIÓN
# ==============================================================================
if __name__ == '__main__':
    Preprocessed_Data_Folder = r"E:\Yoyi\Preprocessed Data"
    Archivo_Salida_CSV = "CCI_RM_ANOVA.csv"

    generate_validation = False  # Cambiar a True si se desean las gráficas de validación
    
    process_metric(Preprocessed_Data_Folder, Archivo_Salida_CSV, plot_validation=generate_validation)

### Statistical Analysis

In [ ]:
import pandas as pd
import pingouin as pg
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ==============================================================================
# FUNCIÓN DE GRAFICADO ADAPTATIVA (SOPORTA 2 O 3 NIVELES TEMPORALES)
# ==============================================================================
def graficar_boxplot(df, metrica, titulo, ylabel):
    niveles_tiempo = [t for t in ['PRE', 'DURING', 'POST'] if t in df['Time'].unique()]
    n_t = len(niveles_tiempo)
    
    fig = plt.figure(figsize=(8 if n_t == 3 else 6, 5 if n_t == 3 else 4))
    
    ax = sns.boxplot(
        data=df, x='Stimulation', y=metrica, hue='Time',
        order=['STIM', 'SHAM'], hue_order=niveles_tiempo,
        color='white', width=0.6 if n_t == 3 else 0.5, gap=0.1 if n_t == 3 else 0.15,
        linewidth=1.5, fliersize=6, medianprops=dict(linewidth=2.5)
    )
    
    for patch in ax.patches:
        patch.set_edgecolor('black')
        patch.set_facecolor('white')

    lineas_medianas = [line for line in ax.lines if line.get_linewidth() == 2.5]
    medianas_ordenadas = sorted(lineas_medianas, key=lambda l: sum(l.get_xdata()) / 2)
    
    if n_t == 3:
        colores_estandar = ['#0072B2', '#E69F00', '#D55E00', '#0072B2', '#E69F00', '#D55E00']
        etiquetas = ['pre', 'dur', 'post', 'pre', 'dur', 'post']
    else:
        colores_estandar = ['#0072B2', '#D55E00', '#0072B2', '#D55E00']
        etiquetas = ['pre', 'post', 'pre', 'post']
        
    for mediana, color in zip(medianas_ordenadas, colores_estandar):
        mediana.set_color(color)

    plt.title(titulo, fontsize=18, fontweight='bold')
    plt.ylabel(ylabel, fontsize=14, fontweight='bold')
    plt.xlabel('')
    plt.xticks(fontsize=12, fontweight='bold')
    
    if ax.legend_:
        ax.legend_.remove()
        
    y_min, y_max = df[metrica].min(), df[metrica].max()
    rango = (y_max - y_min) if (y_max != y_min) else 1.0
    offset_y = rango * 0.08
    
    cajas = ax.patches[:(2 * n_t)]
    cajas_ordenadas = sorted(cajas, key=lambda p: p.get_path().get_extents().bounds[0])
    posiciones_x = [p.get_path().get_extents().bounds[0] + (p.get_path().get_extents().bounds[2] / 2) for p in cajas_ordenadas]
    
    for x, etiqueta, color_txt in zip(posiciones_x, etiquetas, colores_estandar):
        ax.text(x, y_min - offset_y, etiqueta, ha='center', va='top', fontweight='bold', fontsize=11, color=color_txt)
        
    plt.ylim(y_min - (rango * 0.15), y_max + (rango * 0.05))
    plt.tight_layout()
    
    output_dir = os.path.join(os.getcwd(), "output")
    os.makedirs(output_dir, exist_ok=True)
    fig.savefig(os.path.join(output_dir, f"{metrica}.png"), dpi=600, bbox_inches='tight')
    plt.show()
    plt.close(fig)

# ==============================================================================
# ANÁLISIS ESTADÍSTICO UNIFICADO (CCI)
# ==============================================================================
def run_statistical_pipeline(csv_filepath, config_metricas=None, usar_log=False, plot_validation=False):
    print("=" * 90)
    print(f"CARGANDO DATOS DESDE: {csv_filepath} | Bloque: Muscle Co-Contraction Index (CCI)")
    print("=" * 90)
    
    try:
        df = pd.read_csv(csv_filepath)
    except FileNotFoundError:
        print(f"Error: Archivo '{csv_filepath}' no encontrado.")
        return
        
    df['Subject'] = df['Subject'].astype(str)
    df['Stimulation'] = df['Stimulation'].astype(str)
    df['Time'] = df['Time'].astype(str)
    
    sujetos_excluidos_manual = ['1', '2', '10']
    df = df[~df['Subject'].isin(sujetos_excluidos_manual)]
    
    n_tiempos = df['Time'].nunique()
    mediciones_esperadas = 2 * n_tiempos
    conteo_mediciones = df.groupby('Subject').size()
    sujetos_completos = conteo_mediciones[conteo_mediciones == mediciones_esperadas].index.tolist()
    df = df[df['Subject'].isin(sujetos_completos)].copy()
    
    print(f"Sujetos analizados con Cross-Over completo (N = {len(sujetos_completos)}, {mediciones_esperadas} bloques): {sorted(sujetos_completos, key=int)}")
    
    df['Condicion_Cruzada'] = df['Stimulation'] + "_" + df['Time']
    
    # Si no se pasa un diccionario específico, analiza todas las columnas CCI presentes
    if config_metricas is None:
        cols_cci = [c for c in df.columns if c.startswith('CCI_')]
        config_metricas = {c: (c.replace('_', ' '), 'CCI (%)') for c in cols_cci}
        
    tabla_normalidad, tabla_anova, tabla_posthoc = [], [], []
    
    for metrica, (titulo, ylabel) in config_metricas.items():
        if metrica not in df.columns:
            continue
            
        try:
            col_log = f"log_{metrica}"
            df[col_log] = np.log10(np.clip(df[metrica], a_min=1e-6, a_max=None))
            col_analisis = col_log if usar_log else metrica
            ylabel_plot = f"Log10 {ylabel}" if usar_log else ylabel
            
            # 1. Generar Boxplot
            if plot_validation:
                graficar_boxplot(df, col_analisis, titulo, ylabel_plot)
            
            # 2. Pruebas de Normalidad
            pvals_raw = pg.normality(data=df, dv=metrica, group='Condicion_Cruzada')['pval'].values
            pvals_log = pg.normality(data=df, dv=col_log, group='Condicion_Cruzada')['pval'].values
            
            tabla_normalidad.append({
                'Metric': metrica,
                'Raw Normal?': all(p > 0.05 for p in pvals_raw),
                'Raw Min p-val': round(min(pvals_raw), 3),
                'Log Normal?': all(p > 0.05 for p in pvals_log),
                'Log Min p-val': round(min(pvals_log), 3)
            })
            
            # 3. rmANOVA
            anova = pg.rm_anova(data=df, dv=col_analisis, within=['Stimulation', 'Time'], subject='Subject', detailed=True)
            ef_time = anova[anova['Source'] == 'Time']
            ef_stim = anova[anova['Source'] == 'Stimulation']
            ef_int = anova[anova['Source'] == 'Stimulation * Time']
            
            tabla_anova.append({
                'Metric': metrica,
                'Time (F, p)': f"{ef_time['F'].values[0]:.3f}, {ef_time['p-unc'].values[0]:.3f}",
                'Stim (F, p)': f"{ef_stim['F'].values[0]:.3f}, {ef_stim['p-unc'].values[0]:.3f}",
                'Interaction (F, p)': f"{ef_int['F'].values[0]:.3f}, {ef_int['p-unc'].values[0]:.3f}"
            })
            
            # 4. Post-Hoc
            ph = pg.pairwise_tests(data=df, dv=col_analisis, within=['Stimulation', 'Time'], subject='Subject', padjust='bonf', parametric=True)
            ph_stim = ph[(ph['Contrast'] == 'Stimulation * Time') & (ph['Stimulation'] == 'STIM')]
            
            for _, row in ph_stim.iterrows():
                tabla_posthoc.append({
                    'Metric': metrica,
                    'Comparison': f"{row['A']} vs {row['B']} (STIM)",
                    'p-corr': round(row['p-corr'], 3),
                    'Hedges g': round(row['hedges'], 3)
                })
                
        except Exception as e:
            print(f"  [!] Error procesando {metrica}: {e}")

    print("\n--- NORMALITY TEST ---")
    print(pd.DataFrame(tabla_normalidad).to_string(index=False))
    print("\n--- rmANOVA RESULTS ---")
    print(pd.DataFrame(tabla_anova).to_string(index=False))
    print("\n--- POST-HOC (STIM CONDITION) ---")
    if tabla_posthoc:
        print(pd.DataFrame(tabla_posthoc).to_string(index=False))
    print("=" * 90)

if __name__ == '__main__':
    ARCHIVO_CSV = "CCI_RM_ANOVA.csv"

    generate_validation = False  # Cambiar a True si se desean las gráficas de validación

    run_statistical_pipeline(ARCHIVO_CSV, usar_log=False, plot_validation=generate_validation)

## 2.4) CMC

In [ ]:
import mne
import numpy as np
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
from scipy.signal import welch, csd

# Canal EMG de referencia principal para la coherencia (0 = primer canal EMG del archivo, o pon el nombre ej. 'ECR' o 'FCR')
CANAL_EMG_REF = 0

def compute_epoch_coherence(eeg_data, emg_data, sfreq, fmin=4.0, fmax=40.0):
    """
    Calcula la Coherencia Cortico-Muscular (Magnitude-Squared Coherence) promediando
    los espectros cruzados (CSD) y auto-espectros (PSD) a través de las épocas.
    eeg_data: array (n_epochs, n_eeg_ch, n_times)
    emg_data: array (n_epochs, n_times) -> 1 canal EMG rectificado
    Retorna: freqs, coh_matrix de shape (n_eeg_ch, n_freqs)
    """
    n_times = eeg_data.shape[-1]
    # Ventana de Welch de 1.0 s (o longitud total si es menor) con 50% de solapamiento
    nperseg = min(int(sfreq * 1.0), n_times)
    noverlap = int(nperseg * 0.5)
    
    # Rectificar EMG (quitar media DC y tomar valor absoluto) para capturar envolvente de disparo
    emg_rect = np.abs(emg_data - np.mean(emg_data, axis=-1, keepdims=True))
    
    # Auto-espectro del EMG: shape (n_epochs, n_freqs)
    freqs, Pyy = welch(emg_rect, fs=sfreq, nperseg=nperseg, noverlap=noverlap, axis=-1)
    Pyy_mean = np.mean(Pyy, axis=0) # Promedio sobre épocas
    
    mask_f = (freqs >= fmin) & (freqs <= fmax)
    freqs_sel = freqs[mask_f]
    Pyy_sel = Pyy_mean[mask_f]
    
    n_eeg_ch = eeg_data.shape[1]
    coh_matrix = np.zeros((n_eeg_ch, len(freqs_sel)))
    
    for ch_idx in range(n_eeg_ch):
        x = eeg_data[:, ch_idx, :]
        _, Pxx = welch(x, fs=sfreq, nperseg=nperseg, noverlap=noverlap, axis=-1)
        _, Pxy = csd(x, emg_rect, fs=sfreq, nperseg=nperseg, noverlap=noverlap, axis=-1)
        
        # Promediar espectros a través de las épocas ANTES de dividir (requisito matemático de coherencia)
        Pxx_sel = np.mean(Pxx, axis=0)[mask_f]
        Pxy_sel = np.mean(Pxy, axis=0)[mask_f]
        
        denom = Pxx_sel * Pyy_sel
        denom[denom == 0] = 1e-12
        coh_matrix[ch_idx, :] = (np.abs(Pxy_sel) ** 2) / denom
        
    return freqs_sel, coh_matrix

def sincronizar_eeg_emg(epochs_eeg, epochs_emg):
    """
    Asegura que los objetos Epochs de EEG y EMG tengan la misma frecuencia de muestreo
    y el mismo número de épocas alineadas.
    """
    sfreq_eeg = epochs_eeg.info['sfreq']
    sfreq_emg = epochs_emg.info['sfreq']
    
    emg_copy = epochs_emg.copy()
    if sfreq_emg != sfreq_eeg:
        emg_copy.resample(sfreq_eeg, verbose=False)
        
    # Alinear cantidad de épocas si difieren por rechazo de artefactos
    n_min = min(len(epochs_eeg), len(emg_copy))
    return epochs_eeg[:n_min], emg_copy[:n_min]

# ==============================================================================
# 0. Función de Generación de Gráficas de Validación (Espectro CMC 4-40 Hz)
# ==============================================================================
def generate_validation_plots(epochs_eeg, epochs_emg, subject_id, condition, time_blk, output_dir):
    """
    Genera gráficas del espectro de Coherencia Cortico-Muscular (4 a 40 Hz) en C3 y C4
    para las 3 ventanas (Relax, Forward, Backward), incluyendo el umbral de significancia (95% CL).
    """
    fig_dir = os.path.join(output_dir, "Validation_Plots_CMC")
    os.makedirs(fig_dir, exist_ok=True)
    
    ch_plot = ['C3', 'C4']
    picks = [ch for ch in ch_plot if ch in epochs_eeg.ch_names]
    if not picks:
        return
        
    sfreq = epochs_eeg.info['sfreq']
    times = epochs_eeg.times
    eeg_data = epochs_eeg.get_data(picks=picks)
    
    emg_idx = epochs_emg.ch_names.index(CANAL_EMG_REF) if isinstance(CANAL_EMG_REF, str) and CANAL_EMG_REF in epochs_emg.ch_names else 0
    emg_data = epochs_emg.get_data()[:, emg_idx, :]
    
    mask_relax = (times >= 3.0) & (times <= 5.0)
    mask_fwd   = (times >= 5.0) & (times <= 7.0)
    mask_bwd   = (times >= 7.5) & (times <= 9.5)
    
    freqs, coh_relax = compute_epoch_coherence(eeg_data[:, :, mask_relax], emg_data[:, mask_relax], sfreq)
    _, coh_fwd       = compute_epoch_coherence(eeg_data[:, :, mask_fwd], emg_data[:, mask_fwd], sfreq)
    _, coh_bwd       = compute_epoch_coherence(eeg_data[:, :, mask_bwd], emg_data[:, mask_bwd], sfreq)
    
    # Límite de confianza al 95% de Halliday et al. (1995): CL = 1 - (0.05)^(1 / (L - 1))
    n_ep = eeg_data.shape[0]
    conf_limit = 1.0 - (0.05 ** (1.0 / max(n_ep - 1, 1)))
    
    fig_cmc, axes_cmc = plt.subplots(1, len(picks), figsize=(6 * len(picks), 5))
    if len(picks) == 1:
        axes_cmc = [axes_cmc]
        
    for i, ch in enumerate(picks):
        ax = axes_cmc[i]
        ax.plot(freqs, coh_relax[i], label='Relax (3-5s)', color='blue', lw=1.5)
        ax.plot(freqs, coh_fwd[i], label='Forward (5-7s)', color='red', lw=1.8)
        ax.plot(freqs, coh_bwd[i], label='Backward (7.5-9.5s)', color='green', lw=1.5)
        ax.axhline(conf_limit, color='black', linestyle='--', lw=1.2, label='95% Conf. Limit')
        
        ax.axvspan(8, 13, color='yellow', alpha=0.2, label='Alpha (8-13 Hz)')
        ax.axvspan(15, 29, color='pink', alpha=0.3, label='Beta (15-29 Hz)')
        
        ax.set_title(f"CMC - {ch} ({condition} - {time_blk})", fontweight='bold')
        ax.set_xlabel("Frequency (Hz)")
        ax.set_ylabel("Magnitude-Squared Coherence")
        ax.grid(True, alpha=0.3)
        if i == 0:
            ax.legend()
            
    plt.tight_layout()
    fig_cmc.savefig(os.path.join(fig_dir, f"CMC_sub-{subject_id}_{condition}_{time_blk}.png"), dpi=300)
    plt.close(fig_cmc)

# ==============================================================================
# 1. Función de Extracción Matemática (CMC Beta 15-29 Hz por Fases)
# ==============================================================================
def extract_metrics(epochs_eeg, epochs_emg):
    """
    Extrae la Coherencia Cortico-Muscular (CMC) promedio en la banda Beta (15-29 Hz)
    para todos los canales EEG en cuatro ventanas:
    - CMC_{ch} (Global: 3.0 a 9.5 s)
    - CMC_RELAX_{ch} (3.0 a 5.0 s)
    - CMC_FWD_{ch} (5.0 a 7.0 s)
    - CMC_BWD_{ch} (7.5 a 9.5 s)
    """
    canales = epochs_eeg.ch_names
    sfreq = epochs_eeg.info['sfreq']
    times = epochs_eeg.times
    eeg_data = epochs_eeg.get_data()
    
    emg_idx = epochs_emg.ch_names.index(CANAL_EMG_REF) if isinstance(CANAL_EMG_REF, str) and CANAL_EMG_REF in epochs_emg.ch_names else 0
    emg_data = epochs_emg.get_data()[:, emg_idx, :]
    
    mask_all   = (times >= 3.0) & (times <= 9.5)
    mask_relax = (times >= 3.0) & (times <= 5.0)
    mask_fwd   = (times >= 5.0) & (times <= 7.0)
    mask_bwd   = (times >= 7.5) & (times <= 9.5)
    
    # Calcular coherencia exclusivamente en la banda Beta (15 - 29 Hz)
    _, coh_all   = compute_epoch_coherence(eeg_data[:, :, mask_all], emg_data[:, mask_all], sfreq, fmin=15.0, fmax=29.0)
    _, coh_relax = compute_epoch_coherence(eeg_data[:, :, mask_relax], emg_data[:, mask_relax], sfreq, fmin=15.0, fmax=29.0)
    _, coh_fwd   = compute_epoch_coherence(eeg_data[:, :, mask_fwd], emg_data[:, mask_fwd], sfreq, fmin=15.0, fmax=29.0)
    _, coh_bwd   = compute_epoch_coherence(eeg_data[:, :, mask_bwd], emg_data[:, mask_bwd], sfreq, fmin=15.0, fmax=29.0)
    
    cmc_all_mean   = np.mean(coh_all, axis=-1)
    cmc_relax_mean = np.mean(coh_relax, axis=-1)
    cmc_fwd_mean   = np.mean(coh_fwd, axis=-1)
    cmc_bwd_mean   = np.mean(coh_bwd, axis=-1)
    
    dict_all   = {f"CMC_{ch}": val for ch, val in zip(canales, cmc_all_mean)}
    dict_relax = {f"CMC_RELAX_{ch}": val for ch, val in zip(canales, cmc_relax_mean)}
    dict_fwd   = {f"CMC_FWD_{ch}": val for ch, val in zip(canales, cmc_fwd_mean)}
    dict_bwd   = {f"CMC_BWD_{ch}": val for ch, val in zip(canales, cmc_bwd_mean)}
    
    return pd.DataFrame([{**dict_all, **dict_relax, **dict_fwd, **dict_bwd}])

# ==============================================================================
# 2. Pipeline Principal (Emparejamiento EEG + EMG y Consolidación)
# ==============================================================================
def process_metric(preprocessed_folder, output_csv, sujetos_a_procesar=None, plot_validation=True):
    print(f"Buscando pares de archivos EEG y EMG en: {preprocessed_folder}")
    
    search_pattern = os.path.join(preprocessed_folder, "*_eeg-epo.fif")
    archivos_eeg = [f for f in glob.glob(search_pattern) if 'blk-basal' not in os.path.basename(f).lower()]
    
    if not archivos_eeg:
        print("No se encontraron archivos *_eeg-epo.fif válidos. Verifica la ruta.")
        return
        
    print(f"Se encontraron {len(archivos_eeg)} bloques EEG. Iniciando cálculo de CMC...\n")
    lista_resultados = []
    base_out_dir = os.path.dirname(os.path.abspath(output_csv)) or os.getcwd()
    
    for eeg_filepath in archivos_eeg:
        filename = os.path.basename(eeg_filepath)
        emg_filepath = eeg_filepath.replace('_eeg-epo.fif', '_emg-epo.fif')
        
        if not os.path.exists(emg_filepath):
            print(f"  [!] No se encontró el archivo EMG par para {filename}. Omitiendo...")
            continue
            
        try:
            partes = filename.split('_')
            subject_id = partes[0].split('-')[1]
            
            if sujetos_a_procesar is not None and subject_id not in sujetos_a_procesar:
                continue
                
            condition = partes[1].split('-')[1].upper()
            
            if 'pre' in filename.lower(): time_blk = 'PRE'
            elif 'during' in filename.lower(): time_blk = 'DURING'
            elif 'post' in filename.lower(): time_blk = 'POST'
            else: continue
                
            print(f"Procesando CMC: Sujeto {subject_id} | {condition} | {time_blk}")
            
            ep_eeg = mne.read_epochs(eeg_filepath, preload=True, verbose=False)
            ep_emg = mne.read_epochs(emg_filepath, preload=True, verbose=False)
            
            # Sincronizar frecuencia de muestreo y número de épocas
            ep_eeg, ep_emg = sincronizar_eeg_emg(ep_eeg, ep_emg)
            
            if plot_validation:
                generate_validation_plots(ep_eeg, ep_emg, subject_id, condition, time_blk, base_out_dir)
                
            df_resumen = extract_metrics(ep_eeg, ep_emg)
            df_resumen['Subject'] = subject_id
            df_resumen['Stimulation'] = condition
            df_resumen['Time'] = time_blk
            
            lista_resultados.append(df_resumen)
            
        # Al final del bloque try...except dentro del bucle for de CCI y CMC:
        except Exception as e:
            print(f"  [!] Error procesando {filename}: {e}")
        finally:
            plt.close('all')
            gc.collect()
            
    print("\nConsolidando tabla maestra de Coherencia Cortico-Muscular...")
    if lista_resultados:
        df_todos = pd.concat(lista_resultados, ignore_index=True)
        df_maestro = df_todos.groupby(['Subject', 'Stimulation', 'Time']).mean(numeric_only=True).reset_index()
        
        cols_metadatos = ['Subject', 'Stimulation', 'Time']
        cols_metricas = [c for c in df_maestro.columns if c not in cols_metadatos]
        df_maestro = df_maestro[cols_metadatos + cols_metricas]
        
        df_maestro.to_csv(output_csv, index=False)
        print(f"\n¡Éxito! Matriz estadística exportada a: {output_csv}")
        print(df_maestro.head())
    else:
        print("\n[!] No se generaron resultados para exportar.")

# ==============================================================================
# EJECUCIÓN
# ==============================================================================
if __name__ == '__main__':
    Preprocessed_Data_Folder = r"E:\Yoyi\Preprocessed Data"
    Archivo_Salida_CSV = "CMC_RM_ANOVA.csv"

    generate_validation = False  # Cambiar a True si se desean las gráficas de validación
    
    process_metric(Preprocessed_Data_Folder, Archivo_Salida_CSV, plot_validation=generate_validation)

### Statistical Analysis

In [ ]:
import pandas as pd
import pingouin as pg
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ==============================================================================
# FUNCIÓN DE GRAFICADO ADAPTATIVA (2 O 3 NIVELES TEMPORALES)
# ==============================================================================
def graficar_boxplot(df, metrica, titulo, ylabel):
    niveles_tiempo = [t for t in ['PRE', 'DURING', 'POST'] if t in df['Time'].unique()]
    n_t = len(niveles_tiempo)
    
    fig = plt.figure(figsize=(8 if n_t == 3 else 6, 5 if n_t == 3 else 4))
    
    ax = sns.boxplot(
        data=df, x='Stimulation', y=metrica, hue='Time',
        order=['STIM', 'SHAM'], hue_order=niveles_tiempo,
        color='white', width=0.6 if n_t == 3 else 0.5, gap=0.1 if n_t == 3 else 0.15,
        linewidth=1.5, fliersize=6, medianprops=dict(linewidth=2.5)
    )
    
    for patch in ax.patches:
        patch.set_edgecolor('black')
        patch.set_facecolor('white')

    lineas_medianas = [line for line in ax.lines if line.get_linewidth() == 2.5]
    medianas_ordenadas = sorted(lineas_medianas, key=lambda l: sum(l.get_xdata()) / 2)
    
    if n_t == 3:
        colores_estandar = ['#0072B2', '#E69F00', '#D55E00', '#0072B2', '#E69F00', '#D55E00']
        etiquetas = ['pre', 'dur', 'post', 'pre', 'dur', 'post']
    else:
        colores_estandar = ['#0072B2', '#D55E00', '#0072B2', '#D55E00']
        etiquetas = ['pre', 'post', 'pre', 'post']
        
    for mediana, color in zip(medianas_ordenadas, colores_estandar):
        mediana.set_color(color)

    plt.title(titulo, fontsize=18, fontweight='bold')
    plt.ylabel(ylabel, fontsize=14, fontweight='bold')
    plt.xlabel('')
    plt.xticks(fontsize=12, fontweight='bold')
    
    if ax.legend_:
        ax.legend_.remove()
        
    y_min, y_max = df[metrica].min(), df[metrica].max()
    rango = (y_max - y_min) if (y_max != y_min) else 1.0
    offset_y = rango * 0.08
    
    cajas = ax.patches[:(2 * n_t)]
    cajas_ordenadas = sorted(cajas, key=lambda p: p.get_path().get_extents().bounds[0])
    posiciones_x = [p.get_path().get_extents().bounds[0] + (p.get_path().get_extents().bounds[2] / 2) for p in cajas_ordenadas]
    
    for x, etiqueta, color_txt in zip(posiciones_x, etiquetas, colores_estandar):
        ax.text(x, y_min - offset_y, etiqueta, ha='center', va='top', fontweight='bold', fontsize=11, color=color_txt)
        
    plt.ylim(y_min - (rango * 0.15), y_max + (rango * 0.05))
    plt.tight_layout()
    
    output_dir = os.path.join(os.getcwd(), "output")
    os.makedirs(output_dir, exist_ok=True)
    fig.savefig(os.path.join(output_dir, f"{metrica}.png"), dpi=600, bbox_inches='tight')
    plt.close(fig)

# ==============================================================================
# ANÁLISIS ESTADÍSTICO UNIFICADO (CMC)
# ==============================================================================
def run_statistical_pipeline(csv_filepath, canales, tipo_metrica="CMC", usar_log=True, plot_validation=False):
    print("=" * 90)
    print(f"CARGANDO DATOS DESDE: {csv_filepath} | Métrica: {tipo_metrica}")
    print("=" * 90)
    
    try:
        df = pd.read_csv(csv_filepath)
    except FileNotFoundError:
        print(f"Error: Archivo '{csv_filepath}' no encontrado.")
        return
        
    df['Subject'] = df['Subject'].astype(str)
    df['Stimulation'] = df['Stimulation'].astype(str)
    df['Time'] = df['Time'].astype(str)
    
    sujetos_excluidos_manual = ['1', '2', '10']
    df = df[~df['Subject'].isin(sujetos_excluidos_manual)]

    n_tiempos = df['Time'].nunique()
    mediciones_esperadas = 2 * n_tiempos
    conteo_mediciones = df.groupby('Subject').size()
    sujetos_completos = conteo_mediciones[conteo_mediciones == mediciones_esperadas].index.tolist()
    df = df[df['Subject'].isin(sujetos_completos)].copy()
    
    print(f"Sujetos analizados con {mediciones_esperadas} bloques completos (N = {len(sujetos_completos)}): {sorted(sujetos_completos, key=int)}")
    
    df['Condicion_Cruzada'] = df['Stimulation'] + "_" + df['Time']
    tabla_normalidad, tabla_anova, tabla_posthoc = [], [], []
    
    for canal in canales:
        metrica_col = f"{tipo_metrica}_{canal}"
        if metrica_col not in df.columns:
            continue
            
        try:
            col_log = f"log_{metrica_col}"
            df[col_log] = np.log10(np.clip(df[metrica_col], a_min=1e-8, a_max=None))
            
            col_analisis = col_log if usar_log else metrica_col
            ylabel_plot = f"Log10 Beta CMC - {canal}" if usar_log else f"Beta CMC - {canal}"
            
            # 1. Generar Boxplot
            if plot_validation:
                graficar_boxplot(df, col_analisis, f"{tipo_metrica} Beta - {canal}", ylabel_plot)
            
            # 2. Pruebas de Normalidad
            pvals_raw = pg.normality(data=df, dv=metrica_col, group='Condicion_Cruzada')['pval'].values
            pvals_log = pg.normality(data=df, dv=col_log, group='Condicion_Cruzada')['pval'].values
            
            tabla_normalidad.append({
                'Channel': canal,
                'Raw Normal?': all(p > 0.05 for p in pvals_raw),
                'Raw Min p-val': round(min(pvals_raw), 3),
                'Log Normal?': all(p > 0.05 for p in pvals_log),
                'Log Min p-val': round(min(pvals_log), 3)
            })
            
            # 3. rmANOVA
            anova = pg.rm_anova(data=df, dv=col_analisis, within=['Stimulation', 'Time'], subject='Subject', detailed=True)
            ef_stim = anova[anova['Source'] == 'Stimulation']
            ef_time = anova[anova['Source'] == 'Time']
            ef_int = anova[anova['Source'] == 'Stimulation * Time']
            
            tabla_anova.append({
                'Channel': canal,
                'Time (F, p)': f"{ef_time['F'].values[0]:.3f}, {ef_time['p-unc'].values[0]:.3f}",
                'Stim (F, p)': f"{ef_stim['F'].values[0]:.3f}, {ef_stim['p-unc'].values[0]:.3f}",
                'Interaction (F, p)': f"{ef_int['F'].values[0]:.3f}, {ef_int['p-unc'].values[0]:.3f}"
            })
            
            # 4. Post-Hoc
            ph = pg.pairwise_tests(data=df, dv=col_analisis, within=['Stimulation', 'Time'], subject='Subject', padjust='bonf', parametric=True)
            ph_stim = ph[(ph['Contrast'] == 'Stimulation * Time') & (ph['Stimulation'] == 'STIM')]
            
            for _, row in ph_stim.iterrows():
                if row['p-corr'] < 0.1 or n_tiempos == 2:
                    tabla_posthoc.append({
                        'Channel': canal,
                        'Comparison': f"{row['A']} vs {row['B']} (STIM)",
                        'p-corr': round(row['p-corr'], 3),
                        'Hedges g': round(row['hedges'], 3)
                    })
                    
        except Exception as e:
            print(f"  [!] Error procesando el canal {canal}: {e}")

    print("\n--- NORMALITY TEST ---")
    print(pd.DataFrame(tabla_normalidad).to_string(index=False))
    print("\n--- rmANOVA RESULTS ---")
    print(pd.DataFrame(tabla_anova).to_string(index=False))
    print("\n--- POST-HOC (STIM CONDITION) ---")
    if tabla_posthoc:
        print(pd.DataFrame(tabla_posthoc).to_string(index=False))
    else:
        print("No post-hoc contrasts reached p < 0.1")
    print("=" * 90)

if __name__ == '__main__':
    ARCHIVO_CSV = "CMC_RM_ANOVA.csv"
    canales_eeg = ['C3', 'C1', 'Cz', 'C2', 'C4', 'CP1', 'CP2', 'Pz']

    generate_validation = False  # Cambiar a True si se desean las gráficas de validación
    
    # Evaluar CMC Global y por ventanas de 2 segundos
    run_statistical_pipeline(ARCHIVO_CSV, canales_eeg, tipo_metrica="CMC", usar_log=True, plot_validation=generate_validation)
    run_statistical_pipeline(ARCHIVO_CSV, canales_eeg, tipo_metrica="CMC_RELAX", usar_log=True, plot_validation=generate_validation)
    run_statistical_pipeline(ARCHIVO_CSV, canales_eeg, tipo_metrica="CMC_FWD", usar_log=True, plot_validation=generate_validation)
    run_statistical_pipeline(ARCHIVO_CSV, canales_eeg, tipo_metrica="CMC_BWD", usar_log=True, plot_validation=generate_validation)